# NSF Funding Analysis (2021–2025)
**DS 5500 — Chenjie Gu**

This notebook analyzes NSF award data from 2021 to 2025, covering:
1. Funding trends over time by program/CFDA category
2. Top-funded institutions and states per year
3. Program Officer award counts
4. Topic modeling on award abstracts using LDA and BERTopic

> **Run cells in order top-to-bottom.** Each section depends on variables defined above it.  
> All plots are saved as PNG files and then compiled into a single PDF at the end.

## 1. Setup & Installations

In [1]:
import sys
!{sys.executable} -m pip install -q "urllib3<2" seaborn plotly pandas numpy matplotlib openpyxl geopandas
!{sys.executable} -m pip install -q nltk gensim pyLDAvis
!{sys.executable} -m pip install -q bertopic sentence-transformers umap-learn hdbscan scikit-learn
!{sys.executable} -m pip install -q pypdf Pillow kaleido
!pip install statsmodels
!pip install google-genai
print("All packages installed.")

All packages installed.


In [2]:
import os, re, ast, warnings, logging
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # non-interactive backend — avoids display errors
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pc
from plotly.subplots import make_subplots

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet',   quiet=True)

import gensim
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
import pyLDAvis
import pyLDAvis.gensim_models

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

warnings.filterwarnings('ignore')
logging.getLogger('gensim').setLevel(logging.ERROR)
logging.getLogger('bertopic').setLevel(logging.ERROR)


plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style='whitegrid')

# ── Output folder for saved plots ─────────────────────────────────────────────
PLOT_DIR = 'plots'
os.makedirs(PLOT_DIR, exist_ok=True)
SAVED_PLOTS = []   # running list — populated by save_fig() below

def save_fig(fig_or_name, name):
    """Save a matplotlib Figure or a Plotly figure to PLOT_DIR and register it."""
    path = os.path.join(PLOT_DIR, f'{name}.png')
    if hasattr(fig_or_name, 'write_image'):           # Plotly
        fig_or_name.write_image(path, width=1200, height=600, scale=2)
    elif hasattr(fig_or_name, 'savefig'):              # Matplotlib Figure
        fig_or_name.savefig(path, dpi=150, bbox_inches='tight')
        plt.close(fig_or_name)
    SAVED_PLOTS.append(path)
    print(f'  Saved → {path}')

print("All imports successful.")
print(f"Plots will be saved to: {os.path.abspath(PLOT_DIR)}")

All imports successful.
Plots will be saved to: /Users/_fin.fish_/Desktop/NEU/DS5500/plots


## 2. Load & Inspect Data

In [3]:
df = pd.read_excel('NSF_21_25_SG.xlsx', engine='openpyxl')
print(f'Loaded {len(df):,} rows')
print(df.dtypes.head(20))

Loaded 6,477 rows
Unnamed: 0.2                        int64
Unnamed: 0.1                        int64
Unnamed: 0                          int64
awd_id                              int64
agcy_id                            object
tran_type                          object
awd_istr_txt                       object
awd_titl_txt                       object
cfda_num                           object
org_code                            int64
po_phone                          float64
po_email                           object
po_sign_block_name                 object
awd_eff_date               datetime64[ns]
awd_exp_date               datetime64[ns]
tot_intn_awd_amt                    int64
awd_amount                          int64
awd_min_amd_letter_date    datetime64[ns]
awd_max_amd_letter_date    datetime64[ns]
abstract                           object
dtype: object


In [4]:
# ── Data Cleaning ─────────────────────────────────────────────────────────────

df['awd_eff_date']   = pd.to_datetime(df['awd_eff_date'],  errors='coerce')
df['awd_exp_date']   = pd.to_datetime(df['awd_exp_date'],  errors='coerce')
df['year']           = df['awd_eff_year'].astype(int)
df['amount']         = pd.to_numeric(df['awd_amount'],      errors='coerce')
df['duration_years'] = ((df['awd_exp_date'] - df['awd_eff_date']).dt.days / 365.25).round(2)
df['institution']    = df['inst.inst_name']
df['state']          = df['inst.inst_state_code']
df['country']        = df['inst.inst_country_name']

# ── Extract PI name from nested JSON-like string ──────────────────────────────
def extract_pi_name(pi_str):
    try:
        cleaned = str(pi_str).replace('None', '"None"')
        records = ast.literal_eval(cleaned)
        for r in records:
            if r.get('pi_role') == 'Principal Investigator':
                return r.get('pi_full_name', '')
        return records[0].get('pi_full_name', '') if records else ''
    except:
        return ''

# ── Extract max obligation amount ────────────────────────────────────────────
def extract_oblg(oblg_str):
    try:
        records = ast.literal_eval(str(oblg_str))
        if records:
            return max(r.get('fund_oblg_amt', 0) for r in records)
    except:
        pass
    return np.nan

df['pi_name']      = df['pi'].apply(extract_pi_name)
df['oblg_amt']     = df['oblg_fy'].apply(extract_oblg)
df['cfda_primary'] = df['cfda_num'].astype(str).str.split(',').str[0].str.strip()

# ── Filter to 2021-2025 ───────────────────────────────────────────────────────
df = df[df['year'].between(2021, 2025)].copy()

print(f'Years: {df.year.min()} – {df.year.max()}')
print(f'Filtered to {len(df):,} awards (2021–2025)')
print(f'Missing amounts: {df.amount.isna().sum()}')
print(df.year.value_counts().sort_index())

Years: 2021 – 2025
Filtered to 6,398 awards (2021–2025)
Missing amounts: 0
year
2021    1278
2022    1441
2023    1462
2024    1194
2025    1023
Name: count, dtype: int64


## 3. Funding Trends Over Time

In [5]:
# ── Total NSF funding per year ────────────────────────────────────────────────
yearly = df.groupby('year')['amount'].sum().reset_index()
yearly.columns = ['Year', 'Total Funding ($)']

fig = px.bar(
    yearly, x='Year', y='Total Funding ($)',
    title='Total NSF Funding per Year (2021–2025)',
    text_auto='.2s'
)
fig.update_traces(marker_color='steelblue')
fig.update_layout(showlegend=False)
save_fig(fig, '01_total_funding_per_year')
fig.show()

  Saved → plots/01_total_funding_per_year.png


In [6]:
# ── Funding by CFDA category ──────────────────────────────────────────────────
df_cfda = df.copy()
df_cfda['cfda_split'] = df_cfda['cfda_num'].astype(str).str.split(',')
df_cfda = df_cfda.explode('cfda_split')
df_cfda['cfda_split'] = df_cfda['cfda_split'].str.strip()
df_cfda['cfda_split'] = df_cfda['cfda_split'].replace('47.070', '47.07')

cfda_names = {
    '47.07':  'Computer & Info Science',
    '47.049': 'Math & Physical Sciences',
    '47.050': 'Geosciences',
    '47.074': 'Biological Sciences',
    '47.075': 'Social Sciences',
    '47.076': 'Education',
    '47.079': 'STEM Education',
    '47.083': 'Office of Integrative Activities',
    '47.041': 'Engineering',
}
df_cfda['cfda_label'] = df_cfda['cfda_split'].map(cfda_names).fillna(df_cfda['cfda_split'])

cfda_year = df_cfda.groupby(['year', 'cfda_label'])['amount'].sum().reset_index()
top_cfda = (
    cfda_year.groupby('cfda_label')['amount']
    .sum().nlargest(6).index.tolist()
)

fig = px.line(
    cfda_year[cfda_year['cfda_label'] == 'Computer & Info Science'],
    x='year', y='amount', color='cfda_label',
    title='Computer & Info Science Funding Over Time',
    labels={'amount': 'Total Funding ($)', 'year': 'Year', 'cfda_label': 'CFDA Category'},
    markers=True
)
save_fig(fig, '02_cfda_funding_trends')
fig.show()

  Saved → plots/02_cfda_funding_trends.png


In [7]:
# ── Treemap: total funding by CFDA ───────────────────────────────────────────
cfda_total = cfda_year.groupby('cfda_label')['amount'].sum().reset_index()

fig = px.treemap(
    cfda_total, path=['cfda_label'], values='amount',
    title='Total NSF Funding by CFDA Category (2021–2025)',
    color='amount', color_continuous_scale='Blues',
    labels={'amount': 'Total Funding ($)'}
)
fig.update_traces(textinfo='label+value+percent root')
save_fig(fig, '04_cfda_treemap')
fig.show()

  Saved → plots/04_cfda_treemap.png


In [8]:
# ── CSE division funding per year ─────────────────────────────────────────────
print('Division value counts:')
print(df['div_abbr'].value_counts())

div_yearly = df.groupby(['year', 'div_abbr']).agg(
    total_funding=('amount', 'sum'),
    num_awards=('awd_id', 'count')
).reset_index()

div_names = {
    'CSE': 'Computer and Information Science and Engineering (CISE)',
    'OAC': 'Advanced Cyberinfrastructure (CISE/OAC)',
    'CNS': 'Computer and Network Systems (CISE/CNS)',
    'CCF': 'Computing and Communication Foundations (CISE/CCF)',
    'IIS': 'Information and Intelligent Systems (CISE/IIS)',
}
div_yearly['division'] = div_yearly['div_abbr'].map(div_names).fillna(div_yearly['div_abbr'])

fig1 = px.line(
    div_yearly, x='year', y='total_funding', color='division',
    markers=True,
    title='NSF CSE Funding by Division per Year',
    labels={'total_funding': 'Total Funding ($)', 'year': 'Year', 'division': 'Division'}
)
fig1.update_xaxes(tickvals=[2021, 2022, 2023, 2024, 2025])
save_fig(fig1, '05_div_funding_per_year')
fig1.show()

fig2 = px.line(
    div_yearly, x='year', y='num_awards', color='division',
    markers=True,
    title='NSF CSE Award Count by Division per Year',
    labels={'num_awards': 'Number of Awards', 'year': 'Year', 'division': 'Division'}
)
fig2.update_xaxes(tickvals=[2021, 2022, 2023, 2024, 2025])
save_fig(fig2, '06_div_awards_per_year')
fig2.show()

Division value counts:
div_abbr
CNS    2166
IIS    1601
CCF    1445
OAC    1186
Name: count, dtype: int64
  Saved → plots/05_div_funding_per_year.png


  Saved → plots/06_div_awards_per_year.png


In [9]:
# ── Division totals (all years combined) ─────────────────────────────────────
div_total = (
    df.groupby('div_abbr').agg(
        total_funding=('amount', 'sum'),
        num_awards=('awd_id', 'count')
    ).reset_index()
    .sort_values('total_funding', ascending=False)
)
div_names2 = {
    'OAC': 'Advanced Cyberinfrastructure (OAC)',
    'CNS': 'Computer and Network Systems (CNS)',
    'CCF': 'Computing and Communication Foundations (CCF)',
    'IIS': 'Information and Intelligent Systems (IIS)',
}
div_total['division'] = div_total['div_abbr'].map(div_names2).fillna(div_total['div_abbr'])

fig1 = px.bar(
    div_total, x='division', y='total_funding',
    title='NSF CSE Total Funding by Division (2021–2025)',
    labels={'total_funding': 'Total Funding ($)', 'division': 'Division'},
    height=500
)
fig1.update_traces(marker_color='steelblue')
fig1.update_layout(xaxis_tickangle=-15)
save_fig(fig1, '07_div_total_funding')
fig1.show()

fig2 = px.bar(
    div_total, x='division', y='num_awards',
    title='NSF CSE Total Awards by Division (2021–2025)',
    labels={'num_awards': 'Number of Awards', 'division': 'Division'},
    height=500
)
fig2.update_traces(marker_color='steelblue')
fig2.update_layout(xaxis_tickangle=-15)
save_fig(fig2, '08_div_total_awards')
fig2.show()

  Saved → plots/07_div_total_funding.png


  Saved → plots/08_div_total_awards.png


In [10]:
print(df.columns.tolist())

print('pgm_ele sample:')
print(df['pgm_ele'].value_counts().head(20))
print('\npgm_ref sample:')
print(df['pgm_ref'].value_counts().head(20))

['Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0', 'awd_id', 'agcy_id', 'tran_type', 'awd_istr_txt', 'awd_titl_txt', 'cfda_num', 'org_code', 'po_phone', 'po_email', 'po_sign_block_name', 'awd_eff_date', 'awd_exp_date', 'tot_intn_awd_amt', 'awd_amount', 'awd_min_amd_letter_date', 'awd_max_amd_letter_date', 'abstract', 'awd_arra_amount', 'dir_abbr', 'org_dir_long_name', 'div_abbr', 'org_div_long_name', 'awd_agcy_code', 'fund_agcy_code', 'pi', 'pgm_ele', 'pgm_ref', 'app_fund', 'oblg_fy', 'por', 'inst.inst_name', 'inst.inst_street_address', 'inst.inst_street_address_2', 'inst.inst_city_name', 'inst.inst_state_code', 'inst.inst_state_name', 'inst.inst_phone_num', 'inst.inst_zip_code', 'inst.inst_country_name', 'inst.cong_dist_code', 'inst.st_cong_dist_code', 'inst.org_lgl_bus_name', 'inst.org_prnt_uei_num', 'inst.org_uei_num', 'perf_inst.perf_inst_name', 'perf_inst.perf_str_addr', 'perf_inst.perf_city_name', 'perf_inst.perf_st_code', 'perf_inst.perf_st_name', 'perf_inst.perf_zip_code', 'perf_in

In [11]:
import ast

# ── Parse pgm_ele into usable program names ───────────────────────────────────
def extract_pgm_names(val):
    try:
        parsed = ast.literal_eval(val)
        return [p['pgm_ele_name'] for p in parsed]
    except:
        return []

df_pgm = df.copy()
df_pgm['pgm_names'] = df_pgm['pgm_ele'].apply(extract_pgm_names)
df_pgm = df_pgm.explode('pgm_names')
df_pgm = df_pgm[df_pgm['pgm_names'].notna() & (df_pgm['pgm_names'] != '')]

# ── Funding by division and program ───────────────────────────────────────────
div_pgm = (
    df_pgm.groupby(['div_abbr', 'pgm_names']).agg(
        total_funding=('amount', 'sum'),
        num_awards=('awd_id', 'count')
    ).reset_index()
)

div_names = {
    'OAC': 'Advanced Cyberinfrastructure (OAC)',
    'CNS': 'Computer and Network Systems (CNS)',
    'CCF': 'Computing and Communication Foundations (CCF)',
    'IIS': 'Information and Intelligent Systems (IIS)',
}
div_pgm['division'] = div_pgm['div_abbr'].map(div_names).fillna(div_pgm['div_abbr'])

top_pgms = (
    div_pgm.groupby(['division', 'pgm_names'])['total_funding']
    .sum()
    .reset_index()
    .sort_values(['division', 'total_funding'], ascending=[True, False])
)
div_pgm_top = div_pgm[div_pgm['pgm_names'].isin(top_pgms['pgm_names'])]

# ── Funding distribution by Division and Program — individual charts ───────────
for metric, ylabel, slug in [
    ('total_funding', 'Total Funding ($)', 'funding'),
    ('num_awards',    'Number of Awards',  'awards'),
]:
    for div_abbr, div_name in div_names.items():
        div_data = div_pgm_top[div_pgm_top['div_abbr'] == div_abbr].sort_values(metric, ascending=False)
        
        if div_data.empty:
            continue

        fig = px.bar(
            div_data,
            x='pgm_names', y=metric,
            title=f'{div_name} — {ylabel} by Program (2021-2025)',
            labels={metric: ylabel, 'pgm_names': 'Program'},
            height=550,
        )
        fig.update_traces(marker_color=pc.qualitative.Light24[list(div_names.keys()).index(div_abbr)])
        fig.update_layout(xaxis_tickangle=-45, xaxis_tickfont_size=9)
        save_fig(fig, f'33_div_pgm_{div_abbr.lower()}_{slug}')
        fig.show()

  Saved → plots/33_div_pgm_oac_funding.png


  Saved → plots/33_div_pgm_cns_funding.png


  Saved → plots/33_div_pgm_ccf_funding.png


  Saved → plots/33_div_pgm_iis_funding.png


  Saved → plots/33_div_pgm_oac_awards.png


  Saved → plots/33_div_pgm_cns_awards.png


  Saved → plots/33_div_pgm_ccf_awards.png


  Saved → plots/33_div_pgm_iis_awards.png


## 4. Institutional & Geographic Analysis

In [12]:
import numpy as np
import plotly.graph_objects as go

# grand totals (denominators)
grand_funding = df['amount'].sum()
grand_awards  = df['awd_id'].count()

TOP_N = 100
TOP_BUBBLE = 50

# ── Institution totals (all schools, used for treemap + ranking) ──────────────
inst_all = (
    df.groupby('institution').agg(
        total_funding=('amount', 'sum'),
        num_awards=('awd_id', 'count')
    ).reset_index()
)
inst_all = inst_all[inst_all['total_funding'] > 0].reset_index(drop=True)

# ── Bar: top 100 schools by funding ───────────────────────────────────────────
inst_total = (
    inst_all.sort_values('total_funding', ascending=False)
    .head(TOP_N).reset_index(drop=True)
)

bar_fund_share = inst_total['total_funding'].sum() / grand_funding * 100
print(f'Top {TOP_N} schools by funding account for {bar_fund_share:.1f}% of total funding.')

fig1 = px.bar(
    inst_total, x='institution', y='total_funding',
    title=(f'Total NSF CSE Funding by Institution — Top {TOP_N} (2021–2025)'
           f'<br><sup>These schools = {bar_fund_share:.1f}% of funding</sup>'),
    labels={'total_funding': 'Total Funding ($)', 'institution': 'Institution'},
    height=600
)
fig1.update_traces(marker_color='steelblue')
fig1.update_layout(xaxis_tickangle=-90, xaxis_tickfont_size=7)
save_fig(fig1, '10_inst_total_bar')
fig1.show()

# ── Treemap: ALL schools ──────────────────────────────────────────────────────
fig2 = px.treemap(
    inst_all, path=['institution'], values='total_funding',
    title='NSF CSE Funding by Institution — Treemap, All Schools (2021–2025)',
    color='total_funding', color_continuous_scale='Blues',
)
fig2.update_traces(textinfo='label+value')
save_fig(fig2, '11_inst_treemap')
fig2.show()

# ── Bubble: top 50 schools by funding ─────────────────────────────────────────
inst_counts = (
    inst_all.sort_values('total_funding', ascending=False)
    .head(TOP_BUBBLE).reset_index(drop=True)
)

bub_fund_share  = inst_counts['total_funding'].sum() / grand_funding * 100
bub_award_share = inst_counts['num_awards'].sum() / grand_awards * 100
print(f'Top {TOP_BUBBLE} schools by funding account for {bub_fund_share:.1f}% of funding '
      f'and {bub_award_share:.1f}% of awards.')

fig3 = px.scatter(
    inst_counts, x='num_awards', y='total_funding',
    size='total_funding', hover_name='institution',
    text='institution',
    title=(f'NSF CSE Institutions — Bubble Chart (Top {TOP_BUBBLE}, 2021–2025)'
           f'<br><sup>These schools = {bub_fund_share:.1f}% of funding, '
           f'{bub_award_share:.1f}% of awards</sup>'),
    labels={'total_funding': 'Total Funding ($)', 'num_awards': 'Number of Awards'},
    color='total_funding', color_continuous_scale='Teal', height=600
)
fig3.update_traces(textposition='top center', textfont_size=7)
save_fig(fig3, '12_inst_bubble')
fig3.show()

Top 100 schools by funding account for 76.7% of total funding.
  Saved → plots/10_inst_total_bar.png


  Saved → plots/11_inst_treemap.png


Top 50 schools by funding account for 56.6% of funding and 50.3% of awards.
  Saved → plots/12_inst_bubble.png


In [13]:
# ══ AWARD-COUNT VERSIONS ══════════════════════════════════════════════════════

# ── Bar: top 100 schools by award count ───────────────────────────────────────
inst_total_aw = (
    inst_all.sort_values('num_awards', ascending=False)
    .head(TOP_N).reset_index(drop=True)
)

bar_award_share = inst_total_aw['num_awards'].sum() / grand_awards * 100
print(f'Top {TOP_N} schools by awards account for {bar_award_share:.1f}% of all awards.')

fig4 = px.bar(
    inst_total_aw, x='institution', y='num_awards',
    title=(f'Total NSF CSE Awards by Institution — Top {TOP_N} (2021–2025)'
           f'<br><sup>These schools = {bar_award_share:.1f}% of awards</sup>'),
    labels={'num_awards': 'Number of Awards', 'institution': 'Institution'},
    height=600
)
fig4.update_traces(marker_color='steelblue')
fig4.update_layout(xaxis_tickangle=-90, xaxis_tickfont_size=7)
save_fig(fig4, '13_inst_total_bar_awards')
fig4.show()

# ── Treemap: ALL schools, sized by award count ────────────────────────────────
fig5 = px.treemap(
    inst_all, path=['institution'], values='num_awards',
    title='NSF CSE Awards by Institution — Treemap, All Schools (2021–2025)',
    color='num_awards', color_continuous_scale='Blues',
)
fig5.update_traces(textinfo='label+value')
save_fig(fig5, '14_inst_treemap_awards')
fig5.show()

# ── Bubble: top 50 schools by award count ─────────────────────────────────────
inst_counts_aw = (
    inst_all.sort_values('num_awards', ascending=False)
    .head(TOP_BUBBLE).reset_index(drop=True)
)

bub_aw_award_share = inst_counts_aw['num_awards'].sum() / grand_awards * 100
bub_aw_fund_share  = inst_counts_aw['total_funding'].sum() / grand_funding * 100
print(f'Top {TOP_BUBBLE} schools by awards account for {bub_aw_award_share:.1f}% of awards '
      f'and {bub_aw_fund_share:.1f}% of funding.')

fig6 = px.scatter(
    inst_counts_aw, x='num_awards', y='total_funding',
    size='num_awards', hover_name='institution',
    text='institution',
    title=(f'NSF CSE Institutions — Bubble Chart by Awards (Top {TOP_BUBBLE}, 2021–2025)'
           f'<br><sup>These schools = {bub_aw_award_share:.1f}% of awards, '
           f'{bub_aw_fund_share:.1f}% of funding</sup>'),
    labels={'total_funding': 'Total Funding ($)', 'num_awards': 'Number of Awards'},
    color='num_awards', color_continuous_scale='Teal', height=600
)
fig6.update_traces(textposition='top center', textfont_size=7)
save_fig(fig6, '15_inst_bubble_awards')
fig6.show()

Top 100 schools by awards account for 71.9% of all awards.
  Saved → plots/13_inst_total_bar_awards.png


  Saved → plots/14_inst_treemap_awards.png


Top 50 schools by awards account for 50.6% of awards and 56.0% of funding.
  Saved → plots/15_inst_bubble_awards.png


### 4b. State Funding Maps with Campus Locations

Choropleth maps of total funding by state for each year, with Alaska and Hawaii
shrunk into insets so the lower 48 fills the frame. Each institution is plotted
as a point at its real campus (or home-city) location, sized by total funding.
Coordinates for all institutions are embedded in the `CAMPUS` lookup below.

In [14]:
# ── State funding maps: all-568 institution coordinates (city/campus level) ───
# CAMPUS maps every NSF institution name to (lat, lon) at its campus or home city.
# All 567 names resolve; coordinates validated within US bounds. A handful of
# tiny nonprofits/consortia are placed at their home-city center (best guess).
import geopandas as gpd
import shapely.affinity as aff

CAMPUS = {
    "ASSISTMENTS FOUNDATION, INC.": (42.2746, -71.8063),
    "AUGUSTA UNIVERSITY RESEARCH INSTITUTE, INC.": (33.4709, -81.9882),
    "Adler Planetarium": (41.8663, -87.6069),
    "Alabama A&M University": (34.783, -86.569),
    "Alabama State University": (32.364, -86.295),
    "Allen Institute": (47.6231, -122.3489),
    "American Indian Higher Education Consortium": (38.8048, -77.0469),
    "American Institutes for Research in the Behavioral Sciences": (38.8816, -77.1043),
    "American Modeling Teachers Association": (33.4255, -111.94),
    "American Museum Natural History": (40.7813, -73.974),
    "American Society For Engineering Education": (38.9072, -77.0369),
    "American University": (38.9375, -77.0889),
    "Amherst College": (42.3709, -72.517),
    "Ann & Robert H. Lurie Children's Hospital of Chicago": (41.8957, -87.6217),
    "Appalachian State University": (36.2143, -81.6857),
    "Applied Computer Security Associates": (39.084, -77.1528),
    "Arizona State University": (33.4242, -111.9281),
    "Arkansas State University Main Campus": (35.842, -90.675),
    "Association of Research Libraries": (38.9072, -77.0369),
    "Auburn University": (32.5934, -85.494),
    "Auburn University at Montgomery": (32.368, -86.173),
    "BROWARD EDUCATION FOUNDATION, INC": (26.1224, -80.1373),
    "Baldwin Wallace University": (41.3756, -81.847),
    "Ball State University": (40.2057, -85.4083),
    "Bard College": (42.0207, -73.912),
    "Barnard College": (40.809, -73.9636),
    "Battelle Memorial Institute": (40.0289, -83.048),
    "Baylor College of Medicine": (29.71, -95.401),
    "Baylor University": (31.5489, -97.1131),
    "Belmont University": (36.133, -86.795),
    "Benedict College": (34.0118, -81.019),
    "Bentley University": (42.3884, -71.22),
    "Berea College": (37.5718, -84.2918),
    "Beth Israel Deaconess Medical Center": (42.3393, -71.1057),
    "Board of Regents, NSHE, obo University of Nevada, Reno": (39.5447, -119.8143),
    "Board of Trustees of Illinois State University": (40.5108, -88.9943),
    "Boise State University": (43.6029, -116.201),
    "Boston College": (42.3355, -71.1685),
    "Boston VA Research Institute, Inc.": (42.332, -71.111),
    "Bowdoin College": (43.9075, -69.9636),
    "Bowie State University": (39.0143, -76.7822),
    "Bowling Green State University": (41.3776, -83.6371),
    "Bradley University": (40.6975, -89.6157),
    "Brandeis University": (42.3654, -71.2597),
    "Brigham & Women's Hospital Inc": (42.3361, -71.1065),
    "Brigham Young University": (40.2518, -111.6493),
    "Brown University": (41.8268, -71.4025),
    "Bryn Mawr College": (40.027, -75.3157),
    "Bucknell University": (40.954, -76.8843),
    "Butler University": (39.8403, -86.1697),
    "CAL POLY HUMBOLDT SPONSORED PROGRAMS FOUNDATION": (40.8665, -124.0828),
    "COMMUNICATION SERVICE FOR THE DEAF, INC.": (43.5446, -96.7311),
    "COMPUTER SCIENCE ALLIANCE": (37.7749, -122.4194),
    "COMPUTER SCIENCE TEACHERS ASSOCIATION, LLC.": (40.7128, -74.006),
    "CSU Fullerton Auxiliary Services Corporation": (33.8829, -117.8854),
    "CSUB Auxiliary for Sponsored Programs Administration": (35.3489, -119.1029),
    "CUNY Borough of Manhattan Community College": (40.7184, -74.0119),
    "CUNY Brooklyn College": (40.631, -73.9536),
    "CUNY City College": (40.82, -73.9493),
    "CUNY College of Staten Island": (40.601, -74.15),
    "CUNY Hunter College": (40.7685, -73.9646),
    "CUNY John Jay College of Criminal Justice": (40.7704, -73.9883),
    "CUNY New York City College of Technology": (40.6954, -73.9874),
    "CUNY Queens College": (40.7367, -73.8203),
    "CUNY York College": (40.7008, -73.7964),
    "Cal Poly Pomona Foundation, Inc.": (34.0586, -117.8218),
    "California Institute of Technology": (34.1377, -118.1253),
    "California Polytechnic State University Foundation": (35.305, -120.6625),
    "California State L A University Auxiliary Services Inc.": (34.0665, -118.1686),
    "California State University San Marcos Corporation": (33.13, -117.1581),
    "California State University-Dominguez Hills Foundation": (33.8636, -118.2578),
    "California State University-Long Beach Foundation": (33.7838, -118.1141),
    "Calvin University": (42.9311, -85.5867),
    "Carleton College": (44.4609, -93.1538),
    "Carnegie Learning": (40.4406, -79.9959),
    "Carnegie Mellon University": (40.4433, -79.9436),
    "Carnegie-Mellon University": (40.4433, -79.9436),
    "Carthage College": (42.581, -87.823),
    "Case Western Reserve University": (41.5043, -81.6084),
    "Catholic University of America": (38.9339, -76.9986),
    "Center for Open Science": (38.0293, -78.4767),
    "Center for the Advancement of Science in Space, Inc.": (28.3922, -80.6077),
    "Center for the Future of Arizona": (33.4484, -112.074),
    "Central Michigan University": (43.5896, -84.7762),
    "Central State University": (39.7117, -83.8888),
    "Central Washington University": (47.004, -120.538),
    "Chapman University": (33.7935, -117.8516),
    "Charleston Southern Univ": (32.976, -80.057),
    "Chicago Public Schools": (41.8781, -87.6298),
    "Chicago State University": (41.718, -87.608),
    "Chico State Enterprises": (39.7285, -121.8375),
    "Christopher Newport University": (37.0628, -76.494),
    "Claflin University": (33.4915, -80.8584),
    "Claremont McKenna College": (34.1017, -117.7075),
    "Clark University": (42.2509, -71.8231),
    "Clarkson University": (44.6644, -74.9986),
    "Clayton College and State University": (33.551, -84.354),
    "Clemson University": (34.6834, -82.8374),
    "Cleveland Clinic Foundation": (41.5034, -81.621),
    "Cleveland State University": (41.5022, -81.6753),
    "Coastal Carolina University": (33.7929, -79.0094),
    "Code for Science and Society Inc": (45.5152, -122.6784),
    "CodeVA": (37.5407, -77.436),
    "Colgate University": (42.8186, -75.5375),
    "College of Charleston": (32.7841, -79.9373),
    "College of Saint Scholastica": (46.8156, -92.1066),
    "College of William and Mary": (37.271, -76.7075),
    "College of the Holy Cross": (42.2384, -71.8088),
    "Colorado College": (38.8485, -104.8226),
    "Colorado School of Mines": (39.751, -105.2227),
    "Colorado State University": (40.5734, -105.0865),
    "Colorado State University-Pueblo": (38.2849, -104.6091),
    "Columbia University": (40.8075, -73.9626),
    "Computing Research Association": (38.9072, -77.0369),
    "Cornell University": (42.4534, -76.4735),
    "Council of Independent Colleges": (38.9072, -77.0369),
    "Cyber Pack Ventures, Inc.": (39.2904, -76.6122),
    "Dana-Farber Cancer Institute": (42.3376, -71.107),
    "Dartmouth College": (43.7044, -72.2887),
    "Davidson College": (35.4994, -80.8484),
    "DePaul University": (41.9244, -87.6557),
    "Delaware State University": (39.1879, -75.5444),
    "Digital Promise Global": (37.8044, -122.2712),
    "Donald Danforth Plant Science Center": (38.6622, -90.4007),
    "Drexel University": (39.9566, -75.1899),
    "Duke University": (36.0014, -78.9382),
    "Duquesne University": (40.4375, -79.9903),
    "E4THEFUTURE, INC.": (42.4584, -71.0662),
    "EASTIE FARM, INC.": (42.3702, -71.0389),
    "EDUCATE MAINE": (43.6591, -70.2568),
    "East Carolina University": (35.6065, -77.3664),
    "Eastern Michigan University": (42.251, -83.6242),
    "Education Development Center": (42.337, -71.2092),
    "Embry-Riddle Aeronautical University": (29.189, -81.049),
    "Emory University": (33.7925, -84.324),
    "Environmental Defense Fund": (38.9072, -77.0369),
    "FOUNDATION FOR PUERTO RICO, INC.": (18.4655, -66.1057),
    "FPF Education and Innovation Foundation": (38.9072, -77.0369),
    "Fashion Institute of Technology": (40.7475, -73.9942),
    "Florida Agricultural and Mechanical University": (30.4263, -84.2876),
    "Florida Atlantic University": (26.3712, -80.1021),
    "Florida Institute of Technology": (28.0658, -80.6242),
    "Florida International University": (25.7565, -80.3739),
    "Florida Polytechnic University": (28.1517, -81.8497),
    "Florida State University": (30.4419, -84.2985),
    "Fordham University": (40.861, -73.8857),
    "Fort Hays State University": (38.8748, -99.3434),
    "Franklin W. Olin College of Engineering": (42.2935, -71.2641),
    "Furman University": (34.9243, -82.4373),
    "Gallaudet University": (38.908, -76.9938),
    "George Mason University": (38.8315, -77.311),
    "George Washington University": (38.8997, -77.048),
    "Georgetown University": (38.9076, -77.0723),
    "Georgia Gwinnett College": (33.9846, -83.9844),
    "Georgia Southern University Research and Service Foundation, Inc": (32.4204, -81.7831),
    "Georgia State University Research Foundation, Inc.": (33.753, -84.3857),
    "Georgia Tech Research Corporation": (33.7756, -84.3963),
    "Gettysburg College": (39.839, -77.237),
    "Goshen College": (41.5614, -85.8344),
    "Grambling State University": (32.5274, -92.714),
    "Grand Valley State University": (42.9637, -85.8889),
    "Grinnell College": (41.7464, -92.7222),
    "Gulf of Maine Research Institute": (43.6489, -70.2533),
    "H. Lee Moffitt Cancer Center and Research Institute Hospital Inc": (28.0593, -82.4203),
    "HISPANIC FEDERATION INC": (40.7128, -74.006),
    "Hamilton College": (43.0518, -75.3838),
    "Hampton University": (37.021, -76.337),
    "Harvard University": (42.377, -71.1167),
    "Harvey Mudd College": (34.1063, -117.7086),
    "Haverford College": (40.0095, -75.306),
    "High Point University": (35.9707, -79.997),
    "Hofstra University": (40.7148, -73.6101),
    "Howard University": (38.922, -77.0197),
    "Hugo W. Moser Research Institute at Kennedy Krieger, Inc.": (39.2976, -76.5916),
    "INSTITUTE FOR PROTEIN INNOVATION, INC.": (42.3393, -71.1057),
    "INTERNET2": (40.7128, -74.006),
    "Illinois Institute of Technology": (41.8349, -87.627),
    "Indiana University": (39.1682, -86.523),
    "International Computer Science Institute": (37.8761, -122.2686),
    "Internet Society": (38.9586, -77.357),
    "Iowa State University": (42.0267, -93.6465),
    "Ithaca College": (42.4223, -76.496),
    "Ivy Tech Community College of Indiana": (39.7684, -86.1581),
    "Jackson State University": (32.2964, -90.207),
    "Jarvis Christian College": (32.498, -95.086),
    "Joan and Sanford I. Weill Medical College of Cornell University": (40.7649, -73.954),
    "Johns Hopkins University": (39.3299, -76.6205),
    "Kansas State University": (39.1974, -96.5847),
    "Kean University": (40.6804, -74.2393),
    "Kennesaw State University Research and Service Foundation": (34.0382, -84.5818),
    "Kent State University": (41.1497, -81.3434),
    "Kentucky Community & Technical College System": (38.0406, -84.5037),
    "Kettering University": (43.0125, -83.7146),
    "Keystone Initiative for Network Based Education and Research": (40.2737, -76.8844),
    "King's College": (41.2454, -75.8813),
    "Knox College": (40.9384, -90.3712),
    "Krell Institute": (41.5868, -93.625),
    "LEARN: Lonestar Education and Research Network": (30.2672, -97.7431),
    "Lafayette College": (40.6986, -75.2098),
    "Lamar University": (30.041, -94.076),
    "Langston University": (35.9426, -97.257),
    "Las Cumbres Observatory Global Telescope Network": (34.4326, -119.8632),
    "Lawrence Technological University": (42.4736, -83.2496),
    "Lehigh University": (40.6069, -75.3782),
    "Lewis and Clark College": (45.451, -122.67),
    "Lincoln University": (39.809, -90.902),
    "Long Island University": (40.689, -73.9745),
    "Louisiana State University": (30.4133, -91.18),
    "Louisiana State University Health Sciences Center": (29.956, -90.078),
    "Loyola Marymount University": (33.9697, -118.4168),
    "Loyola University of Chicago": (41.9996, -87.6573),
    "Loyola University of Chicago, Health Sciences Campus": (42.0419, -87.8344),
    "MARCONI SOCIETY, INC.": (37.7749, -122.4194),
    "METADATA GAME CHANGERS LLC": (42.2808, -83.743),
    "METROLAB NETWORK, INC.": (38.9072, -77.0369),
    "MGH Institute of Health Professions": (42.3793, -71.0651),
    "MULTIPLIER": (37.7749, -122.4194),
    "Macalester College": (44.9395, -93.169),
    "Manhattan College": (40.8895, -73.9009),
    "Marconi Society, Inc.": (37.7749, -122.4194),
    "Marquette University": (43.0389, -87.9305),
    "Marshall University Research Corporation": (38.4243, -82.428),
    "Marymount University": (38.8965, -77.0708),
    "Massachusetts Institute of Technology": (42.3601, -71.0942),
    "Mayo Clinic Rochester": (44.0225, -92.4669),
    "Meharry Medical College": (36.1664, -86.805),
    "Mercer University": (32.829, -83.649),
    "Merit Network, Inc.": (42.278, -83.7382),
    "Metropolitan State University of Denver": (39.7447, -105.0048),
    "Miami University": (39.507, -84.7349),
    "Michigan State University": (42.7018, -84.4822),
    "Michigan Technological University": (47.119, -88.5476),
    "Middle Tennessee State University": (35.8489, -86.365),
    "Mississippi State University": (33.4552, -88.7944),
    "Mississippi Valley State University": (33.5066, -90.3454),
    "Missouri State University": (37.1985, -93.2796),
    "Missouri University of Science and Technology": (37.9555, -91.7765),
    "Missouri Western State University": (39.7611, -94.7916),
    "Monmouth University": (40.279, -74.004),
    "Montana State University": (45.6677, -111.0498),
    "Montclair State University": (40.8612, -74.198),
    "Morehouse College": (33.746, -84.4137),
    "Morgan State University": (39.3439, -76.5849),
    "Morgridge Institute for Research, Inc.": (43.0747, -89.4189),
    "Mount Holyoke College": (42.2546, -72.574),
    "NJEDge.Net": (40.2206, -74.7563),
    "National Academy of Sciences": (38.8926, -77.0469),
    "National Center For State Courts": (37.271, -76.7075),
    "National Center for Genome Resources": (35.687, -105.9378),
    "NatureServe": (39.7392, -104.9903),
    "Navajo Technical University": (35.7547, -108.1455),
    "Nevada System of Higher Education": (39.5447, -119.8143),
    "Nevada System of Higher Education, Desert Research Institute": (39.5703, -119.8047),
    "New Jersey Institute of Technology": (40.7424, -74.1785),
    "New Mexico Institute of Mining and Technology": (34.0664, -106.9058),
    "New Mexico State University": (32.2825, -106.7494),
    "New York Institute of Technology": (40.7681, -73.6536),
    "New York University": (40.7295, -73.9965),
    "New York University Medical Center": (40.7421, -73.974),
    "Norfolk State University": (36.848, -76.262),
    "North Carolina Agricultural & Technical State University": (36.0759, -79.774),
    "North Carolina Central University": (35.9764, -78.8986),
    "North Carolina State University": (35.7847, -78.6821),
    "North Dakota State University Fargo": (46.8978, -96.8016),
    "Northeastern Illinois University": (41.9803, -87.7173),
    "Northeastern University": (42.3398, -71.0892),
    "Northern Arizona University": (35.1846, -111.6543),
    "Northern Illinois University": (41.9342, -88.774),
    "Northern Kentucky University": (39.0319, -84.4658),
    "Northwestern University": (42.0565, -87.6753),
    "Northwestern University at Chicago": (41.8955, -87.6213),
    "Nova Southeastern University": (26.0786, -80.2412),
    "OHIO STATE UNIVERSITY, THE": (40.0067, -83.0305),
    "OKLAHOMA STATE REGENTS FOR HIGHER EDUCATION": (35.4676, -97.5164),
    "Oakland University": (42.6722, -83.215),
    "Oberlin College": (41.2939, -82.2218),
    "Ohio University": (39.3242, -82.1013),
    "Oklahoma State University": (36.127, -97.0737),
    "Old Dominion University Research Foundation": (36.8855, -76.3058),
    "Oral Roberts University": (36.054, -95.9501),
    "Oregon Health & Science University": (45.4992, -122.6855),
    "Oregon State University": (44.5638, -123.2794),
    "PARKVIEW HOSPITAL, INC.": (41.119, -85.129),
    "PECAN STREET INC.": (30.2672, -97.7431),
    "Pace University New York Campus": (40.7111, -74.005),
    "Pacific American Foundation": (21.395, -157.7395),
    "Palo Alto Veterans Institute for Research": (37.4419, -122.143),
    "Pennsylvania State Univ University Park": (40.7982, -77.8599),
    "Pomona College": (34.0967, -117.7117),
    "Portland State University": (45.5118, -122.685),
    "Prairie View A & M University": (30.0916, -95.988),
    "Princeton University": (40.3431, -74.6551),
    "Purdue University": (40.4237, -86.9212),
    "RFCUNY d/b/a CUNY Grad School of Public Health & Health Policy": (40.7484, -73.984),
    "Rand Corporation": (34.0106, -118.492),
    "Reed College": (45.4807, -122.6307),
    "Regents of the University of Idaho": (46.7263, -117.0118),
    "Regents of the University of Michigan - Ann Arbor": (42.278, -83.7382),
    "Regents of the University of Michigan - Dearborn": (42.3173, -83.233),
    "Regents of the University of Michigan - Flint": (43.0148, -83.6878),
    "Rensselaer Polytechnic Institute": (42.7298, -73.6788),
    "Research Foundation CUNY - Advanced Science Research Center": (40.8186, -73.949),
    "Research Foundation For Mental Hygiene Inc": (42.6526, -73.7562),
    "Research Triangle Institute": (35.9132, -78.8638),
    "Rhode Island Hospital": (41.8092, -71.4072),
    "Rhode Island School of Design": (41.8268, -71.4025),
    "Rice University": (29.7174, -95.4018),
    "Rider University": (40.2823, -74.7404),
    "Rochester Institute of Tech": (43.084, -77.676),
    "Rockefeller University": (40.7625, -73.9568),
    "Rowan University": (39.709, -75.1183),
    "Rutgers University Camden": (39.9489, -75.1242),
    "Rutgers University New Brunswick": (40.5008, -74.4474),
    "Rutgers University Newark": (40.7416, -74.1743),
    "SRI International": (37.457, -122.176),
    "SUNY College at Old Westbury": (40.7884, -73.5926),
    "SUNY College of Environmental Science and Forestry": (43.0345, -76.1366),
    "SUNY Polytechnic Institute": (43.1009, -75.2327),
    "SUNY at Albany": (42.6864, -73.8235),
    "SUNY at Binghamton": (42.0887, -75.9698),
    "SUNY at Buffalo": (43.0008, -78.789),
    "SUNY at Stony Brook": (40.9123, -73.1234),
    "SageFox Consulting Group, LLC": (42.3868, -72.5301),
    "Saint John's University": (45.5816, -94.3919),
    "Saint Lawrence University": (44.5959, -75.1685),
    "Saint Louis University": (38.636, -90.2351),
    "Saint Vincent College": (40.2939, -79.387),
    "Salisbury University": (38.3486, -75.6097),
    "Sam Houston State University": (30.713, -95.55),
    "San Diego State University Foundation": (32.7757, -117.0719),
    "San Francisco State University": (37.7241, -122.4799),
    "San Jacinto College District": (29.691, -95.075),
    "San Jose State University Foundation": (37.3352, -121.8811),
    "Santa Clara University": (37.3496, -121.939),
    "Santa Fe Institute": (35.706, -105.9105),
    "Savannah State University": (32.0233, -81.0707),
    "School Board of Broward County Florida": (26.1224, -80.1373),
    "Scripps College": (34.1037, -117.71),
    "Seattle University": (47.6101, -122.3193),
    "Seton Hall University": (40.7434, -74.2462),
    "Shepherd University": (39.431, -77.805),
    "Skidmore College": (43.0954, -73.7896),
    "Smith College": (42.3185, -72.6406),
    "Sonoma State University": (38.3396, -122.6747),
    "South Dakota Board of Regents": (44.3683, -100.351),
    "South Dakota School of Mines and Technology": (44.0743, -103.2046),
    "South Dakota State University": (44.3193, -96.7837),
    "Southern Illinois University at Carbondale": (37.71, -89.2206),
    "Southern Methodist University": (32.8412, -96.7845),
    "Southern Oregon University": (42.1857, -122.6553),
    "Southern University": (30.5238, -91.1857),
    "Spaulding Rehabilitation Hospital": (42.3794, -71.0651),
    "St Joseph's University": (39.993, -75.244),
    "St Mary's University San Antonio": (29.452, -98.561),
    "Stanford University": (37.4275, -122.1697),
    "Stetson University": (29.0349, -81.3037),
    "Stevens Institute of Technology": (40.7448, -74.0252),
    "Stillman College": (33.211, -87.576),
    "Sustainable Horizons Institute": (37.8716, -122.2727),
    "Swarthmore College": (39.9046, -75.3534),
    "Syracuse University": (43.0392, -76.1351),
    "TERC Inc": (42.3736, -71.1097),
    "THE ALLEN INSTITUTE FOR ARTIFICIAL INTELLIGENCE": (47.6231, -122.3489),
    "TRANSCEND INC": (40.7357, -74.1724),
    "Tarleton State University": (32.2157, -98.2178),
    "Teachers College, Columbia University": (40.81, -73.9601),
    "Temple University": (39.9812, -75.1554),
    "Tennessee State University": (36.1658, -86.8295),
    "Tennessee Technological University": (36.1748, -85.505),
    "Texas A&M AgriLife Research": (30.6188, -96.3365),
    "Texas A&M Engineering Experiment Station": (30.6188, -96.3365),
    "Texas A&M International University": (27.567, -99.438),
    "Texas A&M University": (30.6188, -96.3365),
    "Texas A&M University - Central Texas": (31.11, -97.728),
    "Texas A&M University Corpus Christi": (27.7117, -97.3245),
    "Texas A&M University-Kingsville": (27.5302, -97.8836),
    "Texas A&M University-San Antonio": (29.317, -98.467),
    "Texas Christian University": (32.7092, -97.3628),
    "Texas College": (32.358, -95.289),
    "Texas Southern University": (29.7233, -95.3608),
    "Texas State University - San Marcos": (29.8884, -97.9384),
    "Texas Tech University": (33.5843, -101.8783),
    "Texas Woman's University": (33.2296, -97.1283),
    "The Alexandria Archive Institute": (37.8044, -122.2712),
    "The College Board": (38.9072, -77.0369),
    "The College of New Jersey": (40.2682, -74.7799),
    "The Learning Partnership": (41.8781, -87.6298),
    "The Methodist Hospital Research Institute": (29.71, -95.401),
    "The New York City Foundation for Computer Science Education": (40.7128, -74.006),
    "The Quilt": (40.7128, -74.006),
    "The Research Institute at Nationwide Children's Hospital": (39.952, -82.987),
    "The Salk Institute for Biological Studies": (32.887, -117.2455),
    "The Texas A&M University System HSC": (30.61, -96.34),
    "The University Corporation, Northridge": (34.2381, -118.5301),
    "The University of Central Florida Board of Trustees": (28.6024, -81.2001),
    "The University of Texas Health Science Center at Houston": (29.71, -95.397),
    "The University of Texas Rio Grande Valley": (26.3074, -98.174),
    "Three3": (35.9132, -79.0558),
    "Towson University": (39.3938, -76.6107),
    "Toyota Technological Institute at Chicago": (41.7857, -87.5997),
    "Trinity College": (41.7476, -72.6907),
    "Trivium Consulting": (40.7128, -74.006),
    "Trustees of Boston University": (42.3505, -71.1054),
    "Tufts University": (42.4075, -71.119),
    "Tulane University": (29.9404, -90.1206),
    "Tuskegee University": (32.43, -85.707),
    "US Ignite, Inc.": (38.9072, -77.0369),
    "USENIX Association": (34.4208, -119.6982),
    "University Corporation For Atmospheric Res": (40.0382, -105.243),
    "University Enterprises Corporation at CSUSB": (34.1839, -117.3267),
    "University Enterprises, Incorporated": (38.5557, -121.4227),
    "University of Akron": (41.0758, -81.5121),
    "University of Alabama Tuscaloosa": (33.2144, -87.5391),
    "University of Alabama at Birmingham": (33.502, -86.809),
    "University of Alabama in Huntsville": (34.725, -86.64),
    "University of Alaska Anchorage Campus": (61.189, -149.826),
    "University of Alaska Fairbanks Campus": (64.8576, -147.826),
    "University of Arizona": (32.2319, -110.9501),
    "University of Arkansas": (36.068, -94.172),
    "University of California - Merced": (37.3661, -120.4244),
    "University of California, Office of the President, Oakland": (37.8044, -122.2712),
    "University of California-Berkeley": (37.8719, -122.2585),
    "University of California-Davis": (38.5382, -121.7617),
    "University of California-Irvine": (33.6405, -117.8443),
    "University of California-Los Angeles": (34.0689, -118.4452),
    "University of California-Riverside": (33.9737, -117.3281),
    "University of California-San Diego": (32.8801, -117.234),
    "University of California-San Diego Scripps Inst of Oceanography": (32.8662, -117.254),
    "University of California-San Francisco": (37.7631, -122.4576),
    "University of California-Santa Barbara": (34.414, -119.8489),
    "University of California-Santa Cruz": (36.9916, -122.0583),
    "University of Central Arkansas": (35.079, -92.459),
    "University of Central Florida Board of Trustees": (28.6024, -81.2001),
    "University of Central Missouri": (38.7626, -93.7374),
    "University of Central Oklahoma": (35.6584, -97.4709),
    "University of Chicago": (41.7886, -87.5987),
    "University of Cincinnati Main Campus": (39.1329, -84.515),
    "University of Colorado at Boulder": (40.0076, -105.2659),
    "University of Colorado at Colorado Springs": (38.8916, -104.7913),
    "University of Colorado at Denver": (39.7447, -104.8378),
    "University of Colorado at Denver-Downtown Campus": (39.7447, -105.0048),
    "University of Connecticut": (41.8077, -72.254),
    "University of Connecticut Health Center": (41.721, -72.7906),
    "University of Dayton": (39.7407, -84.1797),
    "University of Delaware": (39.678, -75.7506),
    "University of Denver": (39.6766, -104.9619),
    "University of Detroit Mercy": (42.4156, -83.1338),
    "University of Florida": (29.6436, -82.3549),
    "University of Georgia": (33.948, -83.3773),
    "University of Georgia Research Foundation Inc": (33.948, -83.3773),
    "University of Hawaii": (21.2969, -157.8171),
    "University of Houston": (29.7199, -95.3422),
    "University of Houston - Clear Lake": (29.5767, -95.0966),
    "University of Houston - Downtown": (29.768, -95.361),
    "University of Illinois at Chicago": (41.8708, -87.6505),
    "University of Illinois at Urbana-Champaign": (40.102, -88.2272),
    "University of Iowa": (41.6627, -91.555),
    "University of Kansas Center for Research Inc": (38.9543, -95.2558),
    "University of Kentucky Research Foundation": (38.0307, -84.504),
    "University of Louisiana at Lafayette": (30.2138, -92.0198),
    "University of Louisville Research Foundation Inc": (38.216, -85.7585),
    "University of Maine": (44.9012, -68.6692),
    "University of Maryland Baltimore County": (39.2556, -76.711),
    "University of Maryland Center for Environmental Sciences": (38.3098, -76.453),
    "University of Maryland at Baltimore": (39.2884, -76.6207),
    "University of Maryland, College Park": (38.9869, -76.9426),
    "University of Massachusetts Amherst": (42.3868, -72.5301),
    "University of Massachusetts Boston": (42.3134, -71.0383),
    "University of Massachusetts Lowell": (42.654, -71.326),
    "University of Massachusetts Medical School": (42.2742, -71.7647),
    "University of Massachusetts, Dartmouth": (41.6298, -70.992),
    "University of Memphis": (35.1186, -89.937),
    "University of Miami": (25.7174, -80.2786),
    "University of Miami School of Medicine": (25.7884, -80.2117),
    "University of Minnesota-Twin Cities": (44.974, -93.2277),
    "University of Mississippi": (34.365, -89.5384),
    "University of Missouri-Columbia": (38.9404, -92.3277),
    "University of Missouri-Kansas City": (39.0347, -94.576),
    "University of Missouri-Saint Louis": (38.7095, -90.3107),
    "University of Montana": (46.8607, -113.9852),
    "University of Nebraska Medical Center": (41.2545, -95.9779),
    "University of Nebraska at Omaha": (41.258, -96.0103),
    "University of Nebraska-Lincoln": (40.8202, -96.7005),
    "University of Nevada Las Vegas": (36.1077, -115.1426),
    "University of New Hampshire": (43.134, -70.9264),
    "University of New Mexico": (35.0844, -106.6198),
    "University of New Orleans": (30.0283, -90.0676),
    "University of North Carolina Greensboro": (36.0689, -79.8104),
    "University of North Carolina at Chapel Hill": (35.9049, -79.0469),
    "University of North Carolina at Charlotte": (35.3072, -80.7352),
    "University of North Carolina at Wilmington": (34.2257, -77.8703),
    "University of North Dakota Main Campus": (47.9225, -97.0727),
    "University of North Florida": (30.27, -81.5092),
    "University of North Texas": (33.2103, -97.149),
    "University of Northern Iowa": (42.5141, -92.463),
    "University of Notre Dame": (41.7052, -86.2353),
    "University of Oklahoma Health Sciences Center": (35.4806, -97.4983),
    "University of Oklahoma Norman Campus": (35.2058, -97.4457),
    "University of Oregon Eugene": (44.0448, -123.0726),
    "University of Pennsylvania": (39.9522, -75.1932),
    "University of Pittsburgh": (40.4444, -79.9608),
    "University of Puerto Rico Mayaguez": (18.211, -67.139),
    "University of Puerto Rico Medical Sciences Campus": (18.397, -66.075),
    "University of Puerto Rico-Rio Piedras": (18.4036, -66.049),
    "University of Rhode Island": (41.486, -71.531),
    "University of Rochester": (43.1287, -77.6277),
    "University of Saint Thomas": (44.9404, -93.1897),
    "University of San Francisco": (37.7766, -122.4509),
    "University of South Alabama": (30.696, -88.178),
    "University of South Carolina at Columbia": (33.9938, -81.0301),
    "University of South Dakota Main Campus": (42.7894, -96.929),
    "University of South Florida": (28.0587, -82.4139),
    "University of Southern California": (34.0224, -118.2851),
    "University of Southern Maine": (43.6591, -70.2568),
    "University of Southern Mississippi": (31.3293, -89.3357),
    "University of Tennessee Chattanooga": (35.0517, -85.2967),
    "University of Tennessee Knoxville": (35.9544, -83.9295),
    "University of Tennessee Space Institute": (35.368, -86.24),
    "University of Texas at Arlington": (32.7299, -97.1131),
    "University of Texas at Austin": (30.2849, -97.7341),
    "University of Texas at Dallas": (32.9857, -96.7501),
    "University of Texas at El Paso": (31.7688, -106.505),
    "University of Texas at San Antonio": (29.5832, -98.6196),
    "University of Texas at Tyler": (32.3163, -95.2629),
    "University of Texas, M.D. Anderson Cancer Center": (29.707, -95.397),
    "University of The Incarnate Word": (29.469, -98.467),
    "University of The Virgin Islands": (18.342, -64.973),
    "University of Toledo": (41.6585, -83.6149),
    "University of Toledo Health Science Campus": (41.609, -83.613),
    "University of Tulsa": (36.1521, -95.9447),
    "University of Utah": (40.7649, -111.8421),
    "University of Vermont & State Agricultural College": (44.479, -73.195),
    "University of Virginia Main Campus": (38.0336, -78.508),
    "University of Washington": (47.6553, -122.3035),
    "University of West Florida": (30.5469, -87.2189),
    "University of West Georgia": (33.572, -85.098),
    "University of Wisconsin-Eau Claire": (44.798, -91.501),
    "University of Wisconsin-Madison": (43.0766, -89.4125),
    "University of Wisconsin-Milwaukee": (43.078, -87.881),
    "University of Wisconsin-Parkside": (42.644, -87.856),
    "University of Wisconsin-River Falls": (44.853, -92.624),
    "University of Wisconsin-Whitewater": (42.834, -88.734),
    "University of Wyoming": (41.3149, -105.5666),
    "University of the District of Columbia": (38.9445, -77.0633),
    "Utah State University": (41.7456, -111.8097),
    "Vanderbilt University": (36.1447, -86.8027),
    "Vanderbilt University Medical Center": (36.1419, -86.8027),
    "Vassar College": (41.6862, -73.8957),
    "Villanova University": (40.0372, -75.3429),
    "Virginia Commonwealth University": (37.5491, -77.452),
    "Virginia Polytechnic Institute and State University": (37.2284, -80.4234),
    "W E Upjohn Institute for Employment Research": (42.2917, -85.5872),
    "Wake Forest University": (36.1349, -80.2782),
    "Washington State University": (46.7319, -117.1542),
    "Washington University": (38.6488, -90.3108),
    "Wayne State University": (42.358, -83.0686),
    "Wellesley College": (42.2964, -71.2925),
    "Wesleyan University": (41.5566, -72.6568),
    "West Virginia State University": (38.376, -81.715),
    "West Virginia University Research Corporation": (39.648, -79.967),
    "Western Michigan University": (42.2828, -85.6118),
    "Western Washington University": (48.734, -122.486),
    "Whitman College": (46.0708, -118.33),
    "Whitworth University": (47.7536, -117.414),
    "Wichita State University": (37.7197, -97.2932),
    "William Marsh Rice University": (29.7174, -95.4018),
    "Williams College": (42.7115, -73.2032),
    "Winston-Salem State University": (36.0907, -80.227),
    "Woods Hole Oceanographic Institution": (41.5246, -70.6731),
    "Woodwell Climate Research Center, Inc.": (41.5515, -70.6148),
    "Worcester Polytechnic Institute": (42.2746, -71.8063),
    "Wright State University": (39.7805, -84.0633),
    "Yale University": (41.3163, -72.9223),
    "Yeshiva University": (40.8501, -73.9293),
}


def campus_coords(name):
    return CAMPUS.get(name, (None, None))   # exact-name lookup

# ── resolve coords per institution; fall back to state centroid if unmatched ──
inst_state = df.groupby(['institution', 'state'])['amount'].sum().reset_index()
cc = inst_state['institution'].apply(campus_coords)
inst_state['lat'] = [c[0] for c in cc]
inst_state['lon'] = [c[1] for c in cc]
matched = inst_state['lat'].notna().sum()
print(f'Matched {matched}/{len(inst_state)} institutions to coordinates '
      f'({matched/len(inst_state)*100:.0f}%).')

gdf = gpd.read_file(
    'https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/'
    'geojson/ne_110m_admin_1_states_provinces.geojson'
)
gdf = gdf[gdf['iso_a2'] == 'US'].copy()
gdf['state'] = gdf['postal'].str.strip()
gdf_proj = gdf.to_crs('EPSG:5070')
cent = gdf_proj.geometry.centroid.to_crs('EPSG:4326')
state_centroid = dict(zip(gdf['state'], zip(cent.x, cent.y)))   # (lon, lat)

def resolve(r):
    if pd.notna(r['lat']):
        return r['lat'], r['lon']
    c = state_centroid.get(r['state'])
    return (c[1], c[0]) if c else (None, None)
res = inst_state.apply(resolve, axis=1, result_type='expand')
inst_state['plat'], inst_state['plon'] = res[0], res[1]

state_year = df.groupby(['year', 'state'])['amount'].sum().reset_index()
vmin, vmax = state_year['amount'].min(), state_year['amount'].max()
years = sorted(state_year['year'].unique())
CONUS = gdf[~gdf['state'].isin(['AK', 'HI'])]
minx, miny, maxx, maxy = CONUS.total_bounds

def inset_params(geom, scale, target_left, target_bottom):
    cx, cy = geom.centroid.x, geom.centroid.y
    scaled = aff.scale(geom, xfact=scale, yfact=scale, origin='centroid')
    b = scaled.bounds
    return scale, cx, cy, target_left - b[0], target_bottom - b[1]

def place_inset_geom(geom, params):
    scale, cx, cy, dx, dy = params
    scaled = geom.apply(lambda g: aff.scale(g, xfact=scale, yfact=scale, origin='centroid'))
    return gpd.GeoSeries([aff.translate(g, xoff=dx, yoff=dy) for g in scaled],
                         index=geom.index, crs=geom.crs)

def transform_pts(lon, lat, params):
    scale, cx, cy, dx, dy = params
    return cx + (lon - cx) * scale + dx, cy + (lat - cy) * scale + dy

AK_SCALE, HI_SCALE = 0.40, 1.0

for yr in years:
    yr_data = state_year[state_year['year'] == yr]
    merged = gdf.merge(yr_data, on='state', how='left')
    fig_map, ax = plt.subplots(1, 1, figsize=(14, 8))

    conus = merged[~merged['state'].isin(['AK', 'HI'])]
    conus.plot(column='amount', ax=ax, cmap='YlOrRd', vmin=vmin, vmax=vmax,
               edgecolor='black', linewidth=0.3, legend=True,
               legend_kwds={'label': 'Total Funding ($)', 'orientation': 'vertical',
                            'shrink': 0.6, 'format': lambda x, _: f'${x/1e6:.0f}M'},
               missing_kwds={'color': 'lightgrey'})

    smax = inst_state['amount'].max()
    def draw(sub):
        if len(sub):
            sizes = (sub['amount'] / smax * 250) + 4
            ax.scatter(sub['px'], sub['py'], s=sizes, color='navy', alpha=0.45,
                       edgecolor='white', linewidth=0.3, zorder=5)

    cp = inst_state[~inst_state['state'].isin(['AK', 'HI'])].copy()
    cp['px'], cp['py'] = cp['plon'], cp['plat']
    cp = cp[cp['px'].between(minx-2, maxx+2) & cp['py'].between(miny-2, maxy+2)]
    draw(cp)

    ak = merged[merged['state'] == 'AK']
    if not ak.empty:
        ak = ak.copy()
        akp = inset_params(ak.geometry.iloc[0], AK_SCALE, minx-3, miny-10)
        ak['geometry'] = place_inset_geom(ak.geometry, akp)
        ak.plot(column='amount', ax=ax, cmap='YlOrRd', vmin=vmin, vmax=vmax,
                edgecolor='black', linewidth=0.3, missing_kwds={'color': 'lightgrey'})
        ap = inst_state[inst_state['state'] == 'AK'].copy()
        if len(ap):
            ap['px'], ap['py'] = transform_pts(ap['plon'], ap['plat'], akp); draw(ap)

    hi = merged[merged['state'] == 'HI']
    if not hi.empty:
        hi = hi.copy()
        hip = inset_params(hi.geometry.iloc[0], HI_SCALE, minx+16, miny-8)
        hi['geometry'] = place_inset_geom(hi.geometry, hip)
        hi.plot(column='amount', ax=ax, cmap='YlOrRd', vmin=vmin, vmax=vmax,
                edgecolor='black', linewidth=0.3, missing_kwds={'color': 'lightgrey'})
        hp = inst_state[inst_state['state'] == 'HI'].copy()
        if len(hp):
            hp['px'], hp['py'] = transform_pts(hp['plon'], hp['plat'], hip); draw(hp)

    ax.set_xlim(minx-3, maxx+2); ax.set_ylim(miny-12, maxy+2)
    ax.set_title(f'NSF CSE Funding by State \u2014 {yr}', fontsize=16, fontweight='bold')
    ax.axis('off'); plt.tight_layout()
    save_fig(fig_map, f'13_state_map_{yr}'); plt.show()


Matched 568/568 institutions to coordinates (100%).
  Saved → plots/13_state_map_2021.png
  Saved → plots/13_state_map_2022.png
  Saved → plots/13_state_map_2023.png
  Saved → plots/13_state_map_2024.png
  Saved → plots/13_state_map_2025.png


## 5. Principal Investigator Analysis

In [15]:
pip install statsmodels

Note: you may need to restart the kernel to use updated packages.


In [16]:
import numpy as np
import plotly.graph_objects as go

# grand totals (denominators)
grand_awards  = df['awd_id'].count()
grand_funding = df['amount'].sum()

TOP_PI = 100
TOP_PI_BUBBLE = 25

# ── Bar: top 100 PIs by award count ───────────────────────────────────────────
pi_total = (
    df.groupby('pi_name')['awd_id']
    .count().reset_index()
    .rename(columns={'awd_id': 'num_awards'})
    .sort_values('num_awards', ascending=False)
    .head(TOP_PI)
    .reset_index(drop=True)
)

pi_award_share = pi_total['num_awards'].sum() / grand_awards * 100
print(f'Top {TOP_PI} PIs by awards account for {pi_award_share:.1f}% of all awards.')

fig1 = px.bar(
    pi_total, x='pi_name', y='num_awards',
    title=(f'Top {TOP_PI} Principal Investigators by Total Awards (2021–2025)'
           f'<br><sup>These PIs = {pi_award_share:.1f}% of awards</sup>'),
    labels={'num_awards': 'Total Awards', 'pi_name': 'Principal Investigator'},
    height=600
)
fig1.update_traces(marker_color='steelblue')
fig1.update_layout(xaxis_tickangle=-90, xaxis_tickfont_size=6,
                   plot_bgcolor='white', paper_bgcolor='white')
save_fig(fig1, '14_pi_awards_bar')
fig1.show()

# ── Bubble: top 50 PIs by award count ─────────────────────────────────────────
pi_bubble = (
    df.groupby('pi_name').agg(
        total_awards=('awd_id', 'count'),
        total_funding=('amount', 'sum')
    ).reset_index()
    .sort_values('total_awards', ascending=False)
    .head(TOP_PI_BUBBLE)
    .reset_index(drop=True)
)

pi_funding_share = pi_bubble['total_funding'].sum() / grand_funding * 100
print(f'Top {TOP_PI_BUBBLE} PIs by awards account for {pi_funding_share:.1f}% of total funding.')

fig3 = px.scatter(
    pi_bubble, x='total_awards', y='total_funding',
    size='total_awards', hover_name='pi_name',
    text='pi_name',
    title=(f'Top {TOP_PI_BUBBLE} Principal Investigators — Bubble Chart (2021-2025)'
           f'<br><sup>These PIs = {pi_funding_share:.1f}% of funding</sup>'),
    labels={'total_funding': 'Total Funding ($)', 'total_awards': 'Number of Awards'},
    color='total_funding', color_continuous_scale='Blues', height=750
)
fig3.update_traces(textposition='top center', textfont_size=7)
save_fig(fig3, '16_pi_bubble')
fig3.show()

Top 100 PIs by awards account for 8.1% of all awards.
  Saved → plots/14_pi_awards_bar.png


Top 25 PIs by awards account for 2.5% of total funding.
  Saved → plots/16_pi_bubble.png


In [17]:
# ── Award Duration Distribution ───────────────────────────────────────────────
df['duration_rounded'] = df['duration_years'].round(0)
avg_duration = df['duration_rounded'].dropna()

print(f'Mean award duration:   {avg_duration.mean():.2f} years')
print(f'Median award duration: {avg_duration.median():.2f} years')
print(df['duration_rounded'].value_counts().sort_index())

fig = px.histogram(
    df, x='duration_rounded',
    title='Distribution of NSF Award Durations',
    labels={'duration_rounded': 'Duration (Years)'},
    color_discrete_sequence=['steelblue'],
    nbins=7
)
fig.update_traces(xbins=dict(start=0, end=7, size=1))
fig.add_vline(x=avg_duration.mean(), line_dash='dash', line_color='red',
              annotation_text=f'Mean: {avg_duration.mean():.1f} yrs')
fig.update_layout(bargap=0.1)
save_fig(fig, '17_award_duration_hist')
fig.show()

Mean award duration:   2.91 years
Median award duration: 3.00 years
duration_rounded
0.0     170
1.0     683
2.0    1159
3.0    2382
4.0    1558
5.0     442
6.0       4
Name: count, dtype: int64
  Saved → plots/17_award_duration_hist.png


In [18]:
# ── Institution → PI hierarchical analysis (top 30 schools) ───────────────────
DARK_COLORS = [
    '#1f3a5f', '#5c1a1a', '#1a4d2e', '#4a1f5c', '#5c3a1a',
    '#1a3a4d', '#3d1a4d', '#4d3a1a', '#1a4d4a', '#4d1a3a',
    '#2c2c54', '#5c2a1a', '#1a2e4d', '#3a4d1a', '#4d1a2a',
]

inst_pi = (
    df.groupby(['institution', 'pi_name']).agg(
        total_funding=('amount', 'sum'),
        total_awards=('awd_id', 'count')
    ).reset_index()
)

# institution totals for both metrics
inst_totals = (
    inst_pi.groupby('institution')
    .agg(inst_total=('total_funding', 'sum'),
         inst_awards=('total_awards', 'sum'))
    .reset_index()
)

# grand totals across ALL schools (denominators for the % share)
grand_funding = df['amount'].sum()
grand_awards  = df['awd_id'].count()

TOP_N = 30


def school_centered_ticks(frame):
    """One tick per institution, centered under its group of bars."""
    positions, labels = [], []
    start = 0
    for inst in frame['institution'].drop_duplicates():
        n = (frame['institution'] == inst).sum()
        positions.append(start + n / 2 - 0.5)
        labels.append(inst)
        start += n
    return positions, labels


# ── Bar: total funding — top 30 schools by funding ────────────────────────────
top_funding = (
    inst_totals.sort_values('inst_total', ascending=False)
    .head(TOP_N).reset_index(drop=True)
)
top_funding['inst_rank'] = top_funding.index

f_share = top_funding['inst_total'].sum() / grand_funding * 100
print(f'Top {TOP_N} schools by funding account for {f_share:.1f}% of total funding.')

inst_pi_funding = (
    inst_pi.merge(top_funding[['institution', 'inst_rank']], on='institution')
    .sort_values(['inst_rank', 'total_funding'], ascending=[True, False])
    .reset_index(drop=True)
)
inst_pi_funding['label'] = inst_pi_funding['institution'] + ' — ' + inst_pi_funding['pi_name']
tickvals, ticktext = school_centered_ticks(inst_pi_funding)
ycap_f = inst_pi_funding['total_funding'].quantile(0.99)

fig1 = px.bar(
    inst_pi_funding,
    x='label', y='total_funding',
    color='institution',
    title=(f'PI-Institution Pairs by Total Funding — Top {TOP_N} Schools '
           f'(2021-2025)<br><sup>These schools = {f_share:.1f}% of funding</sup>'),
    labels={'total_funding': 'Total Funding ($)', 'label': 'Institution'},
    height=650,
    color_discrete_sequence=DARK_COLORS
)
fig1.update_layout(
    xaxis=dict(tickmode='array', tickvals=tickvals, ticktext=ticktext,
               tickangle=-90, tickfont_size=8),
    showlegend=False,
    bargap=0
)
save_fig(fig1, '31_inst_pi_funding')
fig1.show()

# ── Bar: total awards — top 30 schools by awards ──────────────────────────────
top_awards = (
    inst_totals.sort_values('inst_awards', ascending=False)
    .head(TOP_N).reset_index(drop=True)
)
top_awards['inst_rank'] = top_awards.index

a_share = top_awards['inst_awards'].sum() / grand_awards * 100
print(f'Top {TOP_N} schools by awards account for {a_share:.1f}% of all awards.')

inst_pi_awards = (
    inst_pi.merge(top_awards[['institution', 'inst_rank']], on='institution')
    .sort_values(['inst_rank', 'total_awards'], ascending=[True, False])
    .reset_index(drop=True)
)
inst_pi_awards['label'] = inst_pi_awards['institution'] + ' — ' + inst_pi_awards['pi_name']
tickvals_a, ticktext_a = school_centered_ticks(inst_pi_awards)
ycap_a = inst_pi_awards['total_awards'].quantile(0.99)

fig2 = px.bar(
    inst_pi_awards,
    x='label', y='total_awards',
    color='institution',
    title=(f'PI-Institution Pairs by Total Awards — Top {TOP_N} Schools '
           f'(2021-2025)<br><sup>These schools = {a_share:.1f}% of awards</sup>'),
    labels={'total_awards': 'Number of Awards', 'label': 'Institution'},
    height=650,
    color_discrete_sequence=DARK_COLORS
)
fig2.update_layout(
    xaxis=dict(tickmode='array', tickvals=tickvals_a, ticktext=ticktext_a,
               tickangle=-90, tickfont_size=8),
    showlegend=False,
    bargap=0
)
save_fig(fig2, '32_inst_pi_awards')
fig2.show()

Top 30 schools by funding account for 41.9% of total funding.
  Saved → plots/31_inst_pi_funding.png


Top 30 schools by awards account for 36.5% of all awards.
  Saved → plots/32_inst_pi_awards.png


## 6. Topic Modeling on Award Abstracts
We apply two methods:
- **LDA** — classical bag-of-words probabilistic model
- **BERTopic** — transformer-based with BERT embeddings + UMAP + HDBSCAN

In [19]:
# ── Inspect abstract column ──────────────────────────────────────────────────
print('=== abstract sample ===')
print(df['abstract'].iloc[0][:300])
print(f'abstract non-null: {df["abstract"].notna().sum()}')


=== abstract sample ===
particle and nuclear physics pnp are fundamentally probabilistic due to quantum mechanics both fields rely on complex montecarlo mcbased simulators that use random number sampling to make predictions for nearly all aspects of experimental design and data interpretation in fact most branches of scien
abstract non-null: 6398


In [20]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Stop words: NLTK base + extended coverage + NSF boilerplate ───────────────
stop_words_cs = set(stopwords.words('english'))
stop_words_cs.update([
    'also', 'yet', 'either', 'neither', 'often', 'usually', 'sometimes',
    'always', 'never', 'ever', 'still', 'already', 'just', 'even', 'though',
    'although', 'since', 'unless', 'whether', 'while', 'within', 'without',
    'among', 'along', 'across', 'around', 'toward', 'towards', 'upon',
    'hereby', 'herein', 'thereof', 'therein', 'whereby', 'wherein',
    'something', 'someone', 'somewhere', 'somehow', 'somewhat',
    'anything', 'anyone', 'anywhere', 'nothing', 'nobody', 'nowhere',
    'everything', 'everyone', 'everywhere', 'each', 'every', 'both',
    'several', 'few', 'less', 'least', 'much', 'more', 'most',
    'per', 'via', 'etc', 'ie', 'eg', 'thus', 'hence', 'therefore',
    'moreover', 'furthermore', 'however', 'nevertheless', 'nonetheless',
    'meanwhile', 'thereafter', 'thereby', 'therefrom',
])
stop_words_cs.update([
    'nsf', 'award', 'funded', 'funding', 'university', 'college',
    'investigator', 'researcher', 'student', 'faculty', 'professor',
    'graduate', 'undergraduate', 'phd', 'postdoc',
    'intellectual', 'merit', 'broader', 'impact',
    'mission', 'reflect', 'criterion', 'deem', 'statutory', 'worthy',
    'research', 'study', 'project', 'program', 'support', 'develop',
    'development', 'provide', 'work', 'result', 'include', 'approach',
    'framework', 'tool', 'application', 'technique', 'method', 'propose',
    'proposed', 'present', 'investigate', 'explore', 'examine', 'address',
    'enable', 'allow', 'improve', 'increase', 'reduce', 'important',
    'critical', 'novel', 'innovative', 'effective', 'efficient', 'robust',
    'scalable', 'real', 'large', 'high', 'broad', 'multiple', 'various',
    'different', 'existing', 'current', 'future', 'potential', 'significant',
    'national', 'foundation', 'science', 'scientific', 'team', 'paper',
    'publication', 'dataset', 'experiment', 'make', 'build', 'create',
    'design', 'show', 'demonstrate', 'achieve', 'perform', 'apply',
    'extend', 'combine', 'integrate', 'leverage', 'utilize', 'deploy',
    'implement', 'system', 'model', 'data', 'information', 'knowledge',
    'task', 'process', 'resource', 'environment', 'platform',
    'infrastructure', 'component', 'layer', 'level', 'step', 'stage',
    'aspect', 'feature', 'property', 'factor', 'element', 'type', 'class',
    'category', 'group', 'number', 'case', 'example', 'instance',
    'scenario', 'condition', 'outcome', 'activity', 'opportunity',
    'effort', 'practice', 'enhance', 'identify', 'require', 'lead',
    'share', 'access', 'plan', 'help', 'need', 'understand', 'exist',
    'base', 'diverse', 'complex', 'service', 'material', 'state',
    'structure', 'benefit', 'distribute', 'change', 'evaluate', 'involve',
    'limit', 'range', 'institution', 'course', 'capability', 'energy',
])

# ── Subfield seed terms (expanded via TF-IDF against abstract corpus) ─────────
cs_subfields_seed = {
    'machine_learning':     ['neural', 'deep learning', 'transformer', 'llm', 'generative',
                             'reinforcement', 'supervised', 'unsupervised', 'embedding',
                             'classification', 'backpropagation', 'convolution', 'lstm',
                             'diffusion', 'finetuning', 'inference', 'adversarial'],
    'cybersecurity':        ['security', 'privacy', 'cryptography', 'malware', 'cyber',
                             'encryption', 'authentication', 'intrusion', 'vulnerability',
                             'firewall', 'forensic', 'breach', 'exploit', 'threat',
                             'blockchain', 'trustworthy', 'identity', 'authorization'],
    'networking':           ['network', 'wireless', '5g', 'protocol', 'routing',
                             'bandwidth', 'latency', 'spectrum', 'communication',
                             'internet', 'edge', 'cloud', 'packet', 'topology',
                             'mimo', 'antenna', 'throughput', 'congestion', 'cellular',
                             'wifi', 'iot', 'sensor'],
    'robotics':             ['robot', 'autonomous', 'drone', 'navigation', 'manipulation',
                             'vehicle', 'planning', 'perception', 'control', 'actuator',
                             'locomotion', 'swarm', 'humanoid', 'teleoperation',
                             'haptic', 'grasping', 'mapping', 'slam', 'motion'],
    'hci':                  ['human computer interaction', 'interface', 'usability',
                             'accessibility', 'visualization', 'augmented reality',
                             'virtual reality', 'wearable', 'gesture', 'experience',
                             'cognitive', 'crowdsourcing', 'gamification', 'collaboration'],
    'algorithms':           ['algorithm', 'complexity', 'optimization', 'graph',
                             'combinatorial', 'approximation', 'randomized', 'streaming',
                             'parallel', 'computational', 'geometry', 'sorting',
                             'dynamic programming', 'heuristic', 'polynomial'],
    'systems':              ['operating system', 'compiler', 'kernel', 'gpu', 'architecture',
                             'memory', 'storage', 'cache', 'processor', 'runtime',
                             'virtualization', 'container', 'performance', 'scalability',
                             'fault tolerance', 'scheduling', 'concurrency', 'fpga', 'cpu'],
    'software_engineering': ['software', 'testing', 'debugging', 'verification', 'devops',
                             'specification', 'refactoring', 'maintenance', 'documentation',
                             'agile', 'deployment', 'bug', 'patch', 'repository',
                             'static analysis', 'formal methods'],
    'data_science':         ['database', 'mining', 'analytics', 'clustering', 'bayesian',
                             'query', 'warehouse', 'knowledge graph', 'ontology',
                             'semantic', 'retrieval', 'indexing', 'anomaly', 'pattern',
                             'statistical', 'causal', 'inference', 'fairness'],
    'quantum_computing':    ['quantum', 'qubit', 'entanglement', 'decoherence', 'circuit',
                             'superposition', 'gate', 'error correction', 'annealing',
                             'simulation', 'speedup', 'supremacy'],
    'cyberinfrastructure':  ['cyberinfrastructure', 'hpc', 'supercomputing', 'workflow',
                             'reproducibility', 'cluster', 'grid', 'pipeline',
                             'openscience', 'provenance', 'interoperability', 'metadata',
                             'repository', 'portal', 'gateway', 'middleware'],
    'health_ai':            ['health', 'clinical', 'genomics', 'bioinformatics', 'biomedical',
                             'medical', 'patient', 'diagnosis', 'treatment', 'disease',
                             'imaging', 'drug', 'electronic health record', 'hospital',
                             'monitoring', 'epidemiology', 'precision medicine'],
}

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
tfidf.fit(df['abstract'].dropna())
vocab = tfidf.get_feature_names_out()

def expand_keywords(seeds, vocab):
    expanded = set(seeds)
    for term in vocab:
        if any(s in term for s in seeds):
            expanded.add(term)
    return list(expanded)

cs_subfields = {
    subfield: expand_keywords(seeds, vocab)
    for subfield, seeds in cs_subfields_seed.items()
}
cs_keywords_keep = set(w for kws in cs_subfields.values() for w in kws)

# ── Preprocessing ─────────────────────────────────────────────────────────────
lemmatizer = WordNetLemmatizer()

def preprocess_cs(text):
    if not isinstance(text, str) or len(text) < 20:
        return []
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t, pos='v') for t in tokens]
    tokens = [lemmatizer.lemmatize(t, pos='n') for t in tokens]
    tokens = [t for t in tokens if t not in stop_words_cs and 3 < len(t) < 25]
    return tokens

def cs_keyword_ratio(tokens):
    if not tokens:
        return 0
    return sum(1 for t in tokens if t in cs_keywords_keep) / len(tokens)

# ── Prepare abstract dataframe only ──────────────────────────────────────────
df_text_abstract = df.copy()
df_text_abstract['tokens'] = df_text_abstract['abstract'].apply(preprocess_cs)
df_text_abstract['cs_ratio'] = df_text_abstract['tokens'].apply(cs_keyword_ratio)
df_text_abstract = df_text_abstract[df_text_abstract['tokens'].map(len) >= 5].reset_index(drop=True)
print(f'Abstract rows after filtering: {len(df_text_abstract):,}')


Abstract rows after filtering: 6,398


### 6a. LDA Topic Modeling

In [21]:
def build_gensim_corpus(tokens):
    """Build gensim dictionary and BoW corpus from a list of token lists."""
    dictionary = corpora.Dictionary(tokens)
    dictionary.filter_extremes(no_below=15, no_above=0.4)
    corpus = [dictionary.doc2bow(t) for t in tokens]
    return dictionary, corpus

def find_best_k(tokens, label, k_range=range(5, 55, 5)):
    """Search k from 5 to 50 in steps of 5 for a thorough sweep."""
    dictionary, corpus = build_gensim_corpus(tokens)
    coherence_scores = []
    for k in k_range:
        lda = LdaModel(corpus=corpus, id2word=dictionary,
                       num_topics=k, random_state=42,
                       passes=15, iterations=150,
                       alpha='auto', eta='auto')
        cm = CoherenceModel(model=lda, texts=tokens,
                            dictionary=dictionary, coherence='c_v')
        score = cm.get_coherence()
        coherence_scores.append(score)
        print(f'  [{label}] k={k:2d}: coherence = {score:.4f}')
    best_k = list(k_range)[coherence_scores.index(max(coherence_scores))]
    print(f'\n  [{label}] Best k = {best_k} (coherence = {max(coherence_scores):.4f})\n')
    return best_k, dictionary, corpus, coherence_scores, list(k_range)

print('=== Abstract ===')
best_k_abstract, dictionary_abs, corpus_abs, scores_abs, k_range_abs = find_best_k(
    df_text_abstract['tokens'].tolist(), 'abstract'
)

fig_coh, ax_coh = plt.subplots(figsize=(10, 4))
ax_coh.plot(k_range_abs, scores_abs, marker='o', label='Abstract', color='steelblue')
ax_coh.set_title('LDA Coherence Score by Number of Topics (Abstract)')
ax_coh.set_xlabel('Number of Topics (k)')
ax_coh.set_ylabel('Coherence Score (c_v)')
ax_coh.legend()
plt.tight_layout()
save_fig(fig_coh, '18_lda_coherence')
plt.show()

print(f'Best k — Abstract: {best_k_abstract}')


=== Abstract ===
  [abstract] k= 5: coherence = 0.3978
  [abstract] k=10: coherence = 0.4744
  [abstract] k=15: coherence = 0.4583
  [abstract] k=20: coherence = 0.4804
  [abstract] k=25: coherence = 0.4739
  [abstract] k=30: coherence = 0.4754
  [abstract] k=35: coherence = 0.4578
  [abstract] k=40: coherence = 0.4504
  [abstract] k=45: coherence = 0.4519
  [abstract] k=50: coherence = 0.4487

  [abstract] Best k = 20 (coherence = 0.4804)

  Saved → plots/18_lda_coherence.png
Best k — Abstract: 20


In [22]:
# ── Train final LDA model using best k from coherence sweep ───────────────────
# best_k_abstract is set automatically from the sweep above.
# Override here if you want a fixed k: best_k_abstract = 20
print(f'Training LDA with k={best_k_abstract}')

lda_abstract = LdaModel(
    corpus=corpus_abs, id2word=dictionary_abs,
    num_topics=best_k_abstract, random_state=42,
    passes=40, iterations=400,
    alpha='auto', eta='auto',
    chunksize=2000, minimum_probability=0.01
)
print('── LDA Abstract Topics ──\n')
for idx, topic in lda_abstract.print_topics(num_words=10):
    words = [w.split('*')[1].replace('"','').strip() for w in topic.split('+')]
    print(f'Topic {idx+1:2d}: {" | ".join(words)}')


Training LDA with k=20
── LDA Abstract Topics ──

Topic  1: wireless | sense | spectrum | communication | device | patient | sensor | health | mobile | monitor
Topic  2: security | attack | secure | vulnerability | cybersecurity | threat | detection | defense | authentication | detect
Topic  3: privacy | event | speech | risk | machine | analysis | technology | private | individual | differential
Topic  4: computational | simulation | engineer | device | circuit | technology | physic | physical | storage | flow
Topic  5: decision | fairness | bias | machine | human | time | mechanism | account | fair | algorithm
Topic  6: image | neural | deep | train | representation | object | computer | vision | network | video
Topic  7: intelligence | artificial | agent | train | decisionmaking | healthcare | ensure | machine | aibased | domain
Topic  8: software | code | test | language | verification | developer | formal | automate | reason | analysis
Topic  9: conference | workshop | participant

In [23]:
# ── Assign dominant topic & CS subfield labels ────────────────────────────────
# IMPORTANT: review the topic words printed above and update this dict to match.
# Keys are 0-indexed topic numbers (0 to best_k_abstract-1).
# The labels below match the k=20 best-k output — re-label if k changed.
def get_dominant_topic(bow, model):
    topics = model.get_document_topics(bow)
    return max(topics, key=lambda x: x[1])[0] if topics else -1

abstract_topic_labels = {
    0:  'Networking & Health Sensing',          # wireless | sense | spectrum | patient | health | sensor
    1:  'Cybersecurity & Threat Detection',     # security | attack | vulnerability | threat | detection
    2:  'Privacy & Fairness',                   # privacy | speech | risk | private | differential
    3:  'Scientific Computing & Simulation',    # computational | simulation | engineer | circuit | physics
    4:  'Fairness & Algorithmic Ethics',        # decision | fairness | bias | fair | algorithm
    5:  'Computer Vision & Deep Learning',      # image | neural | deep | object | vision | video
    6:  'AI Agents & Healthcare',               # intelligence | artificial | agent | healthcare | AI-based
    7:  'Software Engineering & Formal Methods',# software | code | verification | formal | automate
    8:  'Academic Community & Outreach',        # conference | workshop | participant | travel | career
    9:  'Biology & Generative AI',              # teacher | generative | biological | molecular | biology
    10: 'Cyberinfrastructure & Open Science',   # cyberinfrastructure | workflow | datasets | open
    11: 'Algorithms, ML & Theory',              # algorithm | graph | theory | optimization | theoretical
    12: 'Social Media & Online Behavior',       # user | social | online | policy | content | behavior
    13: 'Robotics & Autonomous Systems',        # robot | autonomous | vehicle | safety | robotics
    14: 'Quantum & NLP',                        # quantum | language | error | compression | recommender
    15: 'STEM Education & Workforce',           # train | education | school | workforce | skill
    16: 'Networking & Edge Computing',          # network | edge | internet | smart | traffic | testbed
    17: 'HCI & Accessibility',                  # virtual | visualization | interaction | reality | accessibility
    18: 'Resilience & Smart Cities',            # disaster | resilience | city | local | response | mobility
    19: 'Systems & Architecture',               # compute | hardware | memory | architecture | cloud | HPC
}

# Pad any extra topics if best_k > 15
for i in range(len(abstract_topic_labels), best_k_abstract):
    abstract_topic_labels[i] = f'Topic {i+1}'

df_text_abstract['lda_topic'] = [
    get_dominant_topic(bow, lda_abstract) for bow in corpus_abs
]
df_text_abstract['cs_subfield'] = df_text_abstract['lda_topic'].map(abstract_topic_labels)

print('Abstract subfield distribution:')
print(df_text_abstract['cs_subfield'].value_counts())


Abstract subfield distribution:
cs_subfield
STEM Education & Workforce               760
Algorithms, ML & Theory                  668
Systems & Architecture                   596
Resilience & Smart Cities                454
Software Engineering & Formal Methods    428
Cybersecurity & Threat Detection         374
Networking & Health Sensing              367
Academic Community & Outreach            340
Robotics & Autonomous Systems            327
Networking & Edge Computing              291
Computer Vision & Deep Learning          287
Social Media & Online Behavior           279
HCI & Accessibility                      265
Scientific Computing & Simulation        255
Cyberinfrastructure & Open Science       231
Fairness & Algorithmic Ethics            167
Quantum & NLP                            122
Biology & Generative AI                   92
Privacy & Fairness                        69
AI Agents & Healthcare                    26
Name: count, dtype: int64


In [24]:
# ── LDA: Awards & funding per subfield (aggregated across all years) ──────────
# With only 5 years of data, per-year trends are noisy — aggregate instead.
subfield_total = (
    df_text_abstract.groupby('cs_subfield').agg(
        num_awards=('awd_id', 'count'),
        total_funding=('amount', 'sum')
    ).reset_index()
)

for metric, xlabel, suffix in [
    ('num_awards',    'Number of Awards',  'awards'),
    ('total_funding', 'Total Funding ($)', 'funding'),
]:
    d = subfield_total.sort_values(metric, ascending=True)

    fig_bar = px.bar(
        d, x=metric, y='cs_subfield', orientation='h',
        title=f'NSF CSE {xlabel} per CS Subfield (2021–2025 total)',
        labels={metric: xlabel, 'cs_subfield': 'CS Subfield'},
        height=600,
    )
    fig_bar.update_traces(marker_color='steelblue')
    fig_bar.update_layout(margin=dict(l=240))
    save_fig(fig_bar, f'19_lda_bar_abstract_{suffix}')
    fig_bar.show()

  Saved → plots/19_lda_bar_abstract_awards.png


  Saved → plots/19_lda_bar_abstract_funding.png


In [25]:
# ── Total funding by LDA subfield ────────────────────────────────────────────
topic_funding = (
    df_text_abstract.groupby('cs_subfield')['amount']
    .sum().reset_index()
    .sort_values('amount', ascending=False)
)
fig = px.bar(
    topic_funding, x='amount', y='cs_subfield', orientation='h',
    title='Total NSF Funding by CS Subfield (2021-2025)',
    labels={'amount': 'Total Funding ($)', 'cs_subfield': 'CS Subfield'},
    height=600
)
fig.update_traces(marker_color='steelblue')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
save_fig(fig, '21_lda_funding_abstract')
fig.show()


  Saved → plots/21_lda_funding_abstract.png


In [26]:
# ── LDA topic distribution (aggregated across all years) ─────────────────────
topic_total = (
    df_text_abstract.groupby('cs_subfield')['awd_id']
    .count().reset_index()
    .rename(columns={'awd_id': 'count'})
    .sort_values('count', ascending=True)
)
fig_bar = px.bar(
    topic_total, x='count', y='cs_subfield', orientation='h',
    title='LDA Topic Distribution',
    labels={'count': 'Number of Awards', 'cs_subfield': 'CS Subfield'},
    height=600
)
fig_bar.update_traces(marker_color='steelblue')
fig_bar.update_layout(margin=dict(l=240))
save_fig(fig_bar, '22_lda_area_abstract')
fig_bar.show()

  Saved → plots/22_lda_area_abstract.png


### 6b. BERTopic — Transformer-Based Topic Modeling

In [27]:
# ── Clean text for BERTopic (raw text, not tokens) ────────────────────────────
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

abstracts_list = df_text_abstract['abstract'].apply(clean_text).tolist()
print(f'Abstract docs: {len(abstracts_list):,}')


Abstract docs: 6,398


In [28]:
# ── BERT Embeddings ───────────────────────────────────────────────────────────
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print('Encoding abstracts...')
embeddings_abs = embedding_model.encode(abstracts_list, show_progress_bar=True, batch_size=64)
print(f'Abstract embeddings shape: {embeddings_abs.shape}')


Encoding abstracts...


Batches:   0%|          | 0/100 [00:00<?, ?it/s]

Abstract embeddings shape: (6398, 384)


In [29]:
# ── UMAP + HDBSCAN + BERTopic ─────────────────────────────────────────────────
umap_model    = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                     metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=30, min_samples=10,
                        metric='euclidean', prediction_data=True)
vectorizer    = CountVectorizer(stop_words='english', min_df=5, ngram_range=(1, 2))

topic_model_abstract = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    top_n_words=10, verbose=True
)

print('Fitting BERTopic on abstracts...')
topics_abs, probs_abs = topic_model_abstract.fit_transform(abstracts_list, embeddings_abs)
df_text_abstract['bert_topic'] = topics_abs
n_topics_abs   = len(topic_model_abstract.get_topic_info()) - 1
n_outliers_abs = (df_text_abstract['bert_topic'] == -1).sum()
print(f'BERTopic (abstract): {n_topics_abs} topics, {n_outliers_abs:,} outliers '
      f'({n_outliers_abs/len(df_text_abstract)*100:.1f}%)')


2026-06-22 13:16:39,836 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Fitting BERTopic on abstracts...


2026-06-22 13:16:51,242 - BERTopic - Dimensionality - Completed ✓
2026-06-22 13:16:51,243 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-22 13:16:51,314 - BERTopic - Cluster - Completed ✓
2026-06-22 13:16:51,316 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-22 13:16:52,576 - BERTopic - Representation - Completed ✓


BERTopic (abstract): 61 topics, 1,612 outliers (25.2%)


In [30]:
# ── BERTopic top-word bar charts ──────────────────────────────────────────────
print('── Top Words per BERTopic — Abstract ──\n')
info = topic_model_abstract.get_topic_info()
for _, row in info[info['Topic'] != -1].head(15).iterrows():
    words = [w for w, _ in topic_model_abstract.get_topic(row['Topic'])]
    print(f"T{row['Topic']:3d} ({row['Count']:4,} docs): {' | '.join(words[:8])}")

fig_bc = topic_model_abstract.visualize_barchart(top_n_topics=15, n_words=8)
fig_bc.update_layout(title='BERTopic Top Words — Abstract')
save_fig(fig_bc, '23_bert_barchart_abstract')
fig_bc.show()


── Top Words per BERTopic — Abstract ──

T  0 ( 286 docs): wireless | spectrum | networks | network | communication | communications | radio | 5g
T  1 ( 243 docs): autonomous | safety | systems | learning | control | driving | vehicles | agents
T  2 ( 214 docs): communities | community | resilience | disaster | food | water | emergency | civic
T  3 ( 182 docs): models | medical | learning | clinical | ai | data | project | health
T  4 ( 173 docs): online | social | media | social media | users | information | news | authentication
T  5 ( 165 docs): memory | computing | performance | systems | hardware | parallel | project | design
T  6 ( 149 docs): fairness | fair | algorithmic | data | causal | learning | algorithms | project
T  7 ( 148 docs): network | internet | cloud | performance | dns | applications | project | networks
T  8 ( 141 docs): cs | teachers | computer science | school | computer | computing | students | science
T  9 ( 134 docs): quantum | quantum computing | computing 

In [31]:
# ── BERTopic funding by topic ─────────────────────────────────────────────────
bt_funding = (
    df_text_abstract[df_text_abstract['bert_topic'] != -1]
    .groupby('bert_topic')['amount']
    .sum().reset_index()
    .sort_values('amount', ascending=False)
    .head(15)
)
bt_funding['topic_label'] = bt_funding['bert_topic'].apply(
    lambda t: f"T{t}: " + topic_model_abstract.get_topic(t)[0][0]
)
fig = px.bar(
    bt_funding, x='amount', y='topic_label', orientation='h',
    title='Top 15 BERTopics by Total NSF Funding',
    labels={'amount': 'Total Funding ($)', 'topic_label': 'BERTopic'},
    height=600
)
fig.update_traces(marker_color='steelblue')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
save_fig(fig, '24_bert_funding_abstract')
fig.show()

  Saved → plots/24_bert_funding_abstract.png


In [32]:
# ── BERTopic top topics by award count ───────────────────────────────────────
top_topics = (
    df_text_abstract[df_text_abstract['bert_topic'] != -1]['bert_topic']
    .value_counts().head(10).index.tolist()
)
topic_total = (
    df_text_abstract[df_text_abstract['bert_topic'].isin(top_topics)]
    .groupby('bert_topic')['awd_id']
    .count().reset_index()
    .rename(columns={'awd_id': 'count'})
)
topic_total['topic_label'] = topic_total['bert_topic'].apply(
    lambda t: f"T{t}: " + topic_model_abstract.get_topic(t)[0][0]
)
topic_total = topic_total.sort_values('count', ascending=True)

fig_bar = px.bar(
    topic_total, x='count', y='topic_label', orientation='h',
    title='BERTopic Top 10 Topics by Award Count',
    labels={'count': 'Number of Awards', 'topic_label': 'Topic'},
    height=550
)
fig_bar.update_traces(marker_color='steelblue')
save_fig(fig_bar, '25_bert_topics_over_time_abstract')
fig_bar.show()

  Saved → plots/25_bert_topics_over_time_abstract.png


In [33]:
# ── Topic validation: top words + top documents per LDA topic ─────────────────
print('=' * 70)
print('LDA TOPIC VALIDATION — Top Words vs Top Documents')
print('=' * 70)

for topic_id in range(best_k_abstract):
    # get top words for this topic
    words = [w.split('*')[1].replace('"','').strip()
             for w in lda_abstract.print_topics(num_words=8)[topic_id][1].split('+')]
    
    label = abstract_topic_labels.get(topic_id, f'Topic {topic_id+1}')
    
    print(f'\nTopic {topic_id+1:2d} | {label}')
    print(f'Top words : {" | ".join(words)}')
    print('Top documents:')
    
    # get probability of this topic for every document
    topic_probs = []
    for i, bow in enumerate(corpus_abs):
        doc_topics = dict(lda_abstract.get_document_topics(bow))
        prob = doc_topics.get(topic_id, 0.0)
        topic_probs.append((i, prob))
    
    # sort by probability and take top 5
    top_docs = sorted(topic_probs, key=lambda x: x[1], reverse=True)[:5]
    
    for rank, (doc_idx, prob) in enumerate(top_docs, 1):
        title = df_text_abstract['awd_titl_txt'].iloc[doc_idx]
        abstract_snippet = df_text_abstract['abstract'].iloc[doc_idx][:150].replace('\n', ' ')
        print(f'  {rank}. [{prob:.2f}] {title}')
        print(f'       {abstract_snippet}...')
    
    print('-' * 70)

LDA TOPIC VALIDATION — Top Words vs Top Documents

Topic  1 | Networking & Health Sensing
Top words : wireless | sense | spectrum | communication | device | patient | sensor | health
Top documents:
  1. [0.92] CIF: Small: Signal Processing and Learning for NOMA Millimeter-Wave Massive MIMO Systems
       with the proliferation of versatile devices and dataconsuming services the quest for spectrum efficiency has led to the combination of three disruptiv...
  2. [0.92] CIF: Small: Signal Processing and Learning for NOMA Millimeter-Wave Massive MIMO Systems
       with the proliferation of versatile devices and dataconsuming services the quest for spectrum efficiency has led to the combination of three disruptiv...
  3. [0.85] Collaborative Research: SWIFT: Nonlinear and Inseparable Radar And Data (NIRAD) Transmission Framework for Pareto Efficient Spectrum Access in Future Wireless Networks
       communications and radar are two major applications of electromagnetic waves which convey i

### 6c. Per-Division LDA Topic Modeling

The global LDA model above pools all CISE abstracts into one topic space. Here we instead
fit a **separate LDA model within each division** (OAC, CNS, CCF, IIS), so each division's
subfields are discovered only from its own awards. This surfaces structure that the pooled
model blurs together — e.g. CCF's theory/algorithms topics vs. IIS's ML/HCI topics.

For each division we reuse the same `preprocess_cs` tokens, run the same coherence sweep
(`find_best_k`), train a final model with the division's best k, and assign each award its
dominant within-division topic.


In [34]:
# ── Per-division LDA: coherence sweep + final model for each division ──────────
# Reuses preprocess_cs tokens (already in df_text_abstract), build_gensim_corpus,
# and find_best_k from the global-LDA section above.

DIVISIONS = ['OAC', 'CNS', 'CCF', 'IIS']

# Smaller per-division corpora → search a tighter k range than the global sweep.
DIV_K_RANGE = range(4, 22, 2)
MIN_DOCS_PER_DIV = 50   # safety floor

div_lda = {}   # div_abbr -> dict with model, dictionary, corpus, best_k, sub_df, topic_words

for div in DIVISIONS:
    sub = df_text_abstract[df_text_abstract['div_abbr'] == div].reset_index(drop=True)
    n = len(sub)
    print('=' * 70)
    print(f'Division {div}: {n:,} abstracts')
    print('=' * 70)

    if n < MIN_DOCS_PER_DIV:
        print(f'  Skipping {div} — fewer than {MIN_DOCS_PER_DIV} docs.\n')
        continue

    tokens = sub['tokens'].tolist()

    # cap the sweep's max k so it never exceeds what the corpus can support
    max_k = min(20, max(4, n // 40))
    k_range = [k for k in DIV_K_RANGE if k <= max_k] or [4]

    best_k, dictionary, corpus, scores, ks = find_best_k(
        tokens, div, k_range=k_range
    )

    lda = LdaModel(
        corpus=corpus, id2word=dictionary,
        num_topics=best_k, random_state=42,
        passes=40, iterations=400,
        alpha='auto', eta='auto',
        chunksize=2000, minimum_probability=0.01,
    )

    print(f'── {div} LDA topics (k={best_k}) ──')
    topic_words = {}
    for idx, topic in lda.print_topics(num_words=10):
        words = [w.split('*')[1].replace('"', '').strip() for w in topic.split('+')]
        topic_words[idx] = words
        print(f'  Topic {idx + 1:2d}: {" | ".join(words)}')
    print()

    div_lda[div] = dict(
        model=lda, dictionary=dictionary, corpus=corpus,
        best_k=best_k, sub_df=sub, topic_words=topic_words,
        scores=scores, ks=ks,
    )

print('Per-division LDA complete for:', list(div_lda.keys()))


Division OAC: 1,186 abstracts
  [OAC] k= 4: coherence = 0.3484
  [OAC] k= 6: coherence = 0.3289
  [OAC] k= 8: coherence = 0.3169
  [OAC] k=10: coherence = 0.3090
  [OAC] k=12: coherence = 0.2947
  [OAC] k=14: coherence = 0.3081
  [OAC] k=16: coherence = 0.3277
  [OAC] k=18: coherence = 0.2952
  [OAC] k=20: coherence = 0.3216

  [OAC] Best k = 4 (coherence = 0.3484)

── OAC LDA topics (k=4) ──
  Topic  1: simulation | computational | quantum | physic | dynamic | division | earth | computer | physical | chemistry
  Topic  2: performance | network | computer | algorithm | compression | analysis | device | scale | storage | simulation
  Topic  3: user | security | scientist | datasets | analysis | open | image | domain | privacy | quality
  Topic  4: network | education | workforce | technology | computational | workshop | professional | skill | experience | campus

Division CNS: 2,166 abstracts
  [CNS] k= 4: coherence = 0.3531
  [CNS] k= 6: coherence = 0.4117
  [CNS] k= 8: coherence = 0.4

In [35]:
# ── Assign dominant within-division topic to each award ───────────────────────
# Produces df_div_topics: one row per award with its division + within-division
# topic id and the top-words string for that topic (used as a provisional label).

def get_dominant_topic(bow, model):
    topics = model.get_document_topics(bow)
    return max(topics, key=lambda x: x[1])[0] if topics else -1

# Optional: human-readable labels per (division, topic_id). Fill in after reading
# the printed top words above. Any (div, tid) not in this dict falls back to its
# top-3 words. Keys are (div_abbr, topic_id) with topic_id 0-indexed.
div_topic_labels = {
    # ('CCF', 0): 'Algorithms & Complexity',
    # ('IIS', 1): 'Computer Vision',
}

rows = []
for div, info in div_lda.items():
    model = info['model']
    sub = info['sub_df']
    tw = info['topic_words']
    for i, bow in enumerate(info['corpus']):
        tid = get_dominant_topic(bow, model)
        if tid == -1:
            label = 'Unassigned'
        else:
            label = div_topic_labels.get(
                (div, tid),
                ' / '.join(tw.get(tid, [])[:3])
            )
        rows.append({
            'awd_id':       sub['awd_id'].iloc[i],
            'year':         sub['year'].iloc[i],
            'amount':       sub['amount'].iloc[i],
            'div_abbr':     div,
            'div_topic_id': tid,
            'div_topic':    f'{div}: {label}',
        })

df_div_topics = pd.DataFrame(rows)
print(f'Assigned within-division topics to {len(df_div_topics):,} awards.')
print()
for div in div_lda:
    print(f'── {div} topic distribution ──')
    print(
        df_div_topics[df_div_topics['div_abbr'] == div]['div_topic']
        .value_counts()
    )
    print()


Assigned within-division topics to 6,398 awards.

── OAC topic distribution ──
div_topic
OAC: network / education / workforce         431
OAC: simulation / computational / quantum    271
OAC: performance / network / computer        266
OAC: user / security / scientist             218
Name: count, dtype: int64

── CNS topic distribution ──
div_topic
CNS: wireless / spectrum / communication        226
CNS: attack / privacy / security                215
CNS: train / compute / cybersecurity            212
CNS: software / compute / device                189
CNS: disaster / resilience / response           154
CNS: workshop / security / conference           141
CNS: control / management / smart               136
CNS: privacy / social / user                    128
CNS: wireless / communication / optimization    106
CNS: cloud / user / construction                101
CNS: internet / security / protocol              84
CNS: computer / education / school               74
CNS: mobility / transport

In [36]:
# ── Visualize per-division topics: funding & award count by within-division topic
# One subplot per division, horizontal bars sorted by total funding.

for div in div_lda:
    d = df_div_topics[df_div_topics['div_abbr'] == div]
    agg = (
        d.groupby('div_topic')
         .agg(total_funding=('amount', 'sum'),
              num_awards=('awd_id', 'count'))
         .reset_index()
         .sort_values('total_funding', ascending=True)
    )
    # strip the "DIV: " prefix for cleaner y labels
    agg['topic_label'] = agg['div_topic'].str.replace(f'{div}: ', '', regex=False)

    fig = make_subplots(
        rows=1, cols=2, shared_yaxes=True,
        subplot_titles=('Total Funding ($M)', 'Number of Awards'),
    )
    fig.add_trace(
        go.Bar(
            x=agg['total_funding'] / 1e6, y=agg['topic_label'],
            orientation='h', marker_color='steelblue',
            name='Funding', showlegend=False,
        ),
        row=1, col=1,
    )
    fig.add_trace(
        go.Bar(
            x=agg['num_awards'], y=agg['topic_label'],
            orientation='h', marker_color='indianred',
            name='Awards', showlegend=False,
        ),
        row=1, col=2,
    )
    full = div_names.get(div, div)
    fig.update_layout(
        title=f'{full} — Within-Division LDA Topics (k={div_lda[div]["best_k"]})',
        height=120 + 36 * len(agg),
        margin=dict(l=220),
    )
    save_fig(fig, f'33_div_lda_{div}')
    fig.show()


  Saved → plots/33_div_lda_OAC.png


  Saved → plots/33_div_lda_CNS.png


  Saved → plots/33_div_lda_CCF.png


  Saved → plots/33_div_lda_IIS.png


In [37]:
# ── Per-division BERTopic ─────────────────────────────────────────────────────
# Reuses embeddings_abs (already computed in the global BERTopic cell) — slices
# them per division instead of re-encoding. Fits a separate BERTopic per division.

DIVISIONS = ['OAC', 'CNS', 'CCF', 'IIS']
MIN_DOCS_BERT = 100   # below this, skip (too few to cluster)

# abstracts + embeddings must be row-aligned with df_text_abstract
assert len(embeddings_abs) == len(df_text_abstract), \
    'embeddings_abs is not aligned with df_text_abstract — rerun the embedding cell.'

div_bertopic = {}   # div_abbr -> dict(model, topics, sub_index)

for div in DIVISIONS:
    mask = (df_text_abstract['div_abbr'] == div).values
    n = int(mask.sum())
    print('=' * 70)
    print(f'Division {div}: {n:,} abstracts')
    print('=' * 70)

    if n < MIN_DOCS_BERT:
        print(f'  Skipping {div} — fewer than {MIN_DOCS_BERT} docs.\n')
        continue

    sub_docs = [abstracts_list[i] for i in np.where(mask)[0]]
    sub_emb  = embeddings_abs[mask]

    # smaller clusters for smaller corpora than the global min_cluster_size=30
    umap_d = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                  metric='cosine', random_state=42)
    hdb_d  = HDBSCAN(min_cluster_size=15, min_samples=5,
                     metric='euclidean', prediction_data=True)
    vec_d  = CountVectorizer(stop_words='english', min_df=3, ngram_range=(1, 2))

    tm = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_d, hdbscan_model=hdb_d,
        vectorizer_model=vec_d, top_n_words=10, verbose=False
    )
    topics_d, _ = tm.fit_transform(sub_docs, sub_emb)

    info = tm.get_topic_info()
    n_top = len(info) - 1
    n_out = sum(1 for t in topics_d if t == -1)
    print(f'  {n_top} topics, {n_out:,} outliers ({n_out/n*100:.1f}%)')
    for _, row in info[info['Topic'] != -1].head(12).iterrows():
        words = [w for w, _ in tm.get_topic(row['Topic'])][:8]
        print(f"    T{row['Topic']:2d} ({row['Count']:4,}): {' | '.join(words)}")
    print()

    # write topic ids back, aligned to df_text_abstract rows for this division
    sub_index = df_text_abstract.index[mask]
    div_bertopic[div] = dict(model=tm, topics=topics_d, sub_index=sub_index)

print('Per-division BERTopic complete for:', list(div_bertopic.keys()))

# ── Assemble a tidy frame: one row per award with its within-division BERTopic ─
rows = []
for div, info in div_bertopic.items():
    tm = info['model']
    for pos, idx in enumerate(info['sub_index']):
        tid = info['topics'][pos]
        if tid == -1:
            label = f'{div}: outlier'
        else:
            top_word = tm.get_topic(tid)[0][0]
            label = f'{div}: T{tid} {top_word}'
        rows.append({
            'awd_id':       df_text_abstract.at[idx, 'awd_id'],
            'year':         df_text_abstract.at[idx, 'year'],
            'amount':       df_text_abstract.at[idx, 'amount'],
            'div_abbr':     div,
            'div_bert_id':  tid,
            'div_bert':     label,
        })

df_div_bertopic = pd.DataFrame(rows)
print(f'\nAssigned within-division BERTopics to {len(df_div_bertopic):,} awards.')

Division OAC: 1,186 abstracts
  27 topics, 119 outliers (10.0%)
    T 0 (  95): data | project | community | earths | processes | disaster | research | science
    T 1 (  84): materials | quantum | molecular | computational | chemistry | project | software | simulations
    T 2 (  81): research | network | institutions | education | cyberinfrastructure | university | regional | campus
    T 3 (  79): security | data | privacy | scientific | project | cybersecurity | research | software
    T 4 (  65): cluster | research | computing | hpc | university | support | computational | gpu
    T 5 (  58): ci | science | program | nsf | research | community | support | researchers
    T 6 (  54): data | facilities | research | scientific | science | reproducibility | project | open
    T 7 (  46): ai | nairr | ci | training | research | project | pilot | cybersecurity
    T 8 (  44): scientific | computing | workflows | network | project | data | resources | workflow
    T 9 (  42): students | 

In [38]:
# ── Visualize per-division BERTopics: funding & award count ───────────────────
for div in div_bertopic:
    d = df_div_bertopic[(df_div_bertopic['div_abbr'] == div) &
                        (df_div_bertopic['div_bert_id'] != -1)]
    if d.empty:
        print(f'{div}: no non-outlier topics to plot.')
        continue

    agg = (
        d.groupby('div_bert')
         .agg(total_funding=('amount', 'sum'), num_awards=('awd_id', 'count'))
         .reset_index()
         .sort_values('total_funding', ascending=True)
    )
    agg['topic_label'] = agg['div_bert'].str.replace(f'{div}: ', '', regex=False)

    fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                        subplot_titles=('Total Funding ($M)', 'Number of Awards'))
    fig.add_trace(go.Bar(x=agg['total_funding'] / 1e6, y=agg['topic_label'],
                         orientation='h', marker_color='steelblue', showlegend=False),
                  row=1, col=1)
    fig.add_trace(go.Bar(x=agg['num_awards'], y=agg['topic_label'],
                         orientation='h', marker_color='indianred', showlegend=False),
                  row=1, col=2)
    full = div_names.get(div, div) if 'div_names' in dir() else div
    fig.update_layout(
        title=f'{full} — Within-Division BERTopics',
        height=140 + 34 * len(agg), margin=dict(l=240)
    )
    save_fig(fig, f'34_div_bert_{div}')
    fig.show()

  Saved → plots/34_div_bert_OAC.png


  Saved → plots/34_div_bert_CNS.png


  Saved → plots/34_div_bert_CCF.png


  Saved → plots/34_div_bert_IIS.png


### 6d. LLM-Based Topic Modeling — moved

The LLM-based topic modeling lives in a **separate notebook**,
`NSF_Funding_Analysis_LLM.ipynb`, so it can be developed independently of the
finished LDA/BERTopic analysis. That notebook reloads the data and rebuilds the
`abstracts_list`, then runs the Gemini discover → assign → analyze pipeline.

## 7. LDA vs. BERTopic Comparison

In [39]:
# ── Heatmap: LDA vs BERTopic assignment overlap ───────────────────────────────
overlap = df_text_abstract[df_text_abstract['bert_topic'] != -1].copy()
overlap['lda_label']  = overlap['cs_subfield']
overlap['bert_label'] = 'BERT_' + overlap['bert_topic'].astype(str)

crosstab = pd.crosstab(overlap['lda_label'], overlap['bert_label']).iloc[:, :20]

fig_ht, ax_ht = plt.subplots(figsize=(18, 8))
sns.heatmap(crosstab, cmap='YlOrRd', linewidths=0.2, annot=False, ax=ax_ht)
ax_ht.set_title('LDA vs BERTopic Assignment Overlap', fontsize=14, fontweight='bold')
ax_ht.set_xlabel('BERTopic')
ax_ht.set_ylabel('LDA CS Subfield')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
save_fig(fig_ht, '27_lda_bert_heatmap_abstract')
plt.show()


  Saved → plots/27_lda_bert_heatmap_abstract.png


### 7a. Per-Division Method Comparison

The heatmaps below cross-tabulate the within-division LDA topics against the
within-division BERTopics for each division. Bright cells mean both methods
grouped the same awards together, so they show where the two approaches agree
on a division's internal structure.

In [40]:
# ── Per-division LDA vs BERTopic comparison heatmaps ──────────────────────────
# Same crosstab idea as the global comparison above, but computed within each
# division so we can see whether the two methods agree on a division's structure.
for div in div_lda:
    if div not in div_bertopic:
        continue
    lda_d  = df_div_topics[df_div_topics['div_abbr'] == div][['awd_id', 'div_topic']]
    bert_d = df_div_bertopic[(df_div_bertopic['div_abbr'] == div) &
                             (df_div_bertopic['div_bert_id'] != -1)][['awd_id', 'div_bert']]
    merged = lda_d.merge(bert_d, on='awd_id', how='inner')
    if merged.empty:
        print(f'{div}: no overlapping non-outlier awards.'); continue
    merged['lda']  = merged['div_topic'].str.replace(f'{div}: ', '', regex=False)
    merged['bert'] = merged['div_bert'].str.replace(f'{div}: ', '', regex=False)
    ct = pd.crosstab(merged['lda'], merged['bert'])

    fig_ht, ax_ht = plt.subplots(figsize=(min(22, 2 + 0.5 * ct.shape[1]),
                                          max(4, 0.5 * ct.shape[0])))
    sns.heatmap(ct, cmap='YlOrRd', linewidths=0.2, ax=ax_ht)
    full = div_names.get(div, div) if 'div_names' in dir() else div
    ax_ht.set_title(f'{full} — LDA vs BERTopic Overlap', fontsize=13, fontweight='bold')
    ax_ht.set_xlabel('Within-Division BERTopic'); ax_ht.set_ylabel('Within-Division LDA Topic')
    plt.xticks(rotation=45, ha='right', fontsize=7); plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    save_fig(fig_ht, f'35_div_lda_bert_overlap_{div}')
    plt.show()


  Saved → plots/35_div_lda_bert_overlap_OAC.png
  Saved → plots/35_div_lda_bert_overlap_CNS.png
  Saved → plots/35_div_lda_bert_overlap_CCF.png
  Saved → plots/35_div_lda_bert_overlap_IIS.png


In [41]:
# ── Subfield distribution bar chart ──────────────────────────────────────────
abs_dist = df_text_abstract['cs_subfield'].value_counts().reset_index()
abs_dist.columns = ['subfield', 'count']
abs_dist['pct'] = abs_dist['count'] / abs_dist['count'].sum() * 100

fig_dist = px.bar(
    abs_dist, x='pct', y='subfield', orientation='h',
    title='NSF CSE Abstract Subfield Distribution (% of Awards)',
    labels={'pct': '% of Awards', 'subfield': 'CS Subfield'},
    height=600
)
fig_dist.update_traces(marker_color='steelblue')
fig_dist.update_layout(yaxis={'categoryorder': 'total ascending'})
save_fig(fig_dist, '28_subfield_distribution')
fig_dist.show()


  Saved → plots/28_subfield_distribution.png


In [42]:
# ── Top growing vs declining subfields ────────────────────────────────────────
subfield_year = (
    df_text_abstract.groupby(['year', 'cs_subfield']).agg(
        num_awards=('awd_id', 'count'),
        total_funding=('amount', 'sum')
    ).reset_index()
)
for metric, ylabel, slug in [
    ('num_awards',    '% Change in Number of Awards', 'awards'),
    ('total_funding', '% Change in Total Funding',    'funding'),
]:
    pivot = subfield_year.pivot(index='cs_subfield', columns='year', values=metric).fillna(0)
    pivot['pct_change'] = (
        (pivot[2025] - pivot[2021]) / pivot[2021].replace(0, np.nan) * 100
    ).round(1)
    pivot = pivot.dropna(subset=['pct_change']).sort_values('pct_change', ascending=False)
    colors = ['green' if x >= 0 else 'red' for x in pivot['pct_change']]

    fig_g, ax_g = plt.subplots(figsize=(12, 7))
    ax_g.barh(pivot.index, pivot['pct_change'], color=colors)
    ax_g.axvline(x=0, color='black', linewidth=0.8, linestyle='--')
    ax_g.set_title(f'CS Subfield Growth 2021-2025 ({ylabel})',
                   fontsize=13, fontweight='bold')
    ax_g.set_xlabel(ylabel)
    plt.tight_layout()
    save_fig(fig_g, f'29_growth_abstract_{slug}')
    plt.show()


  Saved → plots/29_growth_abstract_awards.png
  Saved → plots/29_growth_abstract_funding.png


In [43]:
# ── Heatmap: subfield x year — Total Funding ($M) ────────────────────────────
subfield_year = (
    df_text_abstract.groupby(['year', 'cs_subfield'])['amount']
    .sum().reset_index()
)
pivot = subfield_year.pivot(
    index='cs_subfield', columns='year', values='amount'
).fillna(0) / 1e6

fig_hm, ax_hm = plt.subplots(figsize=(12, 8))
sns.heatmap(pivot, cmap='YlOrRd', annot=True, fmt='.1f',
            linewidths=0.3, cbar_kws={'label': 'Funding ($M)'}, ax=ax_hm)
ax_hm.set_title('NSF CSE Funding by CS Subfield and Year ($M)',
                fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig(fig_hm, '30_subfield_year_heatmap_abstract')
plt.show()


  Saved → plots/30_subfield_year_heatmap_abstract.png


In [44]:
# ── Heatmap: CS Subfield x Year — Number of Awards ───────────────────────────
subfield_year = (
    df_text_abstract.groupby(['year', 'cs_subfield'])['awd_id']
    .count().reset_index()
    .rename(columns={'awd_id': 'num_awards'})
)
pivot = subfield_year.pivot(
    index='cs_subfield', columns='year', values='num_awards'
).fillna(0).astype(int)
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

fig = px.imshow(
    pivot,
    text_auto=True,
    color_continuous_scale='Blues',
    aspect='auto',
    title='NSF CSE Number of Awards per CS Subfield per Year',
    labels={'x': 'Year', 'y': 'CS Subfield', 'color': 'Number of Awards'},
)
fig.update_layout(
    height=100 + 60 * len(pivot),
    xaxis=dict(tickvals=list(pivot.columns)),
    yaxis=dict(tickfont=dict(size=11)),
    coloraxis_colorbar=dict(title='Awards'),
)
fig.update_traces(textfont=dict(size=11))
save_fig(fig, '30_subfield_awards_heatmap_abstract')
fig.show()


  Saved → plots/30_subfield_awards_heatmap_abstract.png


## 8. Save Results

In [45]:
# ── Save enriched CSV dataset ─────────────────────────────────────────────────
out_cols = ['awd_id', 'awd_titl_txt', 'year', 'amount', 'duration_years',
            'institution', 'state', 'div_abbr', 'pi_name',
            'lda_topic', 'cs_subfield', 'bert_topic']

df_text_abstract[[c for c in out_cols if c in df_text_abstract.columns]].to_csv(
    'nsf_awards_abstract_topics.csv', index=False
)
print('Saved: nsf_awards_abstract_topics.csv')


Saved: nsf_awards_abstract_topics.csv


In [46]:
# ── Save LDA & BERTopic models ────────────────────────────────────────────────
lda_abstract.save('lda_abstract_model')
print('LDA model saved')

topic_model_abstract.save(
    'bertopic_abstract_model', serialization='safetensors', save_ctfidf=True
)
print('BERTopic model saved')


LDA model saved
BERTopic model saved


## 9. Export All Plots to PDF

In [47]:
from PIL import Image
from pypdf import PdfWriter, PdfReader
import io

print(f'Plots registered: {len(SAVED_PLOTS)}')
for p in SAVED_PLOTS:
    exists = os.path.exists(p)
    print(f'  {p}  ← {"OK" if exists else "MISSING"}')

Plots registered: 57
  plots/01_total_funding_per_year.png  ← OK
  plots/02_cfda_funding_trends.png  ← OK
  plots/04_cfda_treemap.png  ← OK
  plots/05_div_funding_per_year.png  ← OK
  plots/06_div_awards_per_year.png  ← OK
  plots/07_div_total_funding.png  ← OK
  plots/08_div_total_awards.png  ← OK
  plots/33_div_pgm_oac_funding.png  ← OK
  plots/33_div_pgm_cns_funding.png  ← OK
  plots/33_div_pgm_ccf_funding.png  ← OK
  plots/33_div_pgm_iis_funding.png  ← OK
  plots/33_div_pgm_oac_awards.png  ← OK
  plots/33_div_pgm_cns_awards.png  ← OK
  plots/33_div_pgm_ccf_awards.png  ← OK
  plots/33_div_pgm_iis_awards.png  ← OK
  plots/10_inst_total_bar.png  ← OK
  plots/11_inst_treemap.png  ← OK
  plots/12_inst_bubble.png  ← OK
  plots/13_inst_total_bar_awards.png  ← OK
  plots/14_inst_treemap_awards.png  ← OK
  plots/15_inst_bubble_awards.png  ← OK
  plots/13_state_map_2021.png  ← OK
  plots/13_state_map_2022.png  ← OK
  plots/13_state_map_2023.png  ← OK
  plots/13_state_map_2024.png  ← OK
  plo

In [48]:
# ── Convert each PNG → single-page PDF, then merge ────────────────────────────
# This approach is robust: no LaTeX, no nbconvert, no browser — pure Python.

PDF_OUTPUT = 'NSF_Funding_Analysis_plots.pdf'
writer     = PdfWriter()
skipped    = []

for png_path in SAVED_PLOTS:
    if not os.path.exists(png_path):
        skipped.append(png_path)
        continue
    try:
        img = Image.open(png_path).convert('RGB')
        page_buf = io.BytesIO()
        # A4 landscape at 150 dpi
        img.save(page_buf, format='PDF', resolution=150)
        page_buf.seek(0)
        reader = PdfReader(page_buf)
        for page in reader.pages:
            writer.add_page(page)
    except Exception as e:
        print(f'  WARNING: could not add {png_path}: {e}')
        skipped.append(png_path)

with open(PDF_OUTPUT, 'wb') as f:
    writer.write(f)

size_mb = os.path.getsize(PDF_OUTPUT) / 1e6
print(f'\nPDF saved → {PDF_OUTPUT}  ({size_mb:.1f} MB, {len(writer.pages)} pages)')
if skipped:
    print(f'Skipped {len(skipped)} missing files: {skipped}')


PDF saved → NSF_Funding_Analysis_plots.pdf  (9.8 MB, 57 pages)
